# Tool 10 — Sleep macrostructure: the classical PSG summary metrics

Computes, for every recording of a database, the metrics that summarise the **architecture of the
night** — the ones printed on a clinical PSG report:

- **time in bed, sleep onset latency, sleep period time, total sleep time, wake after sleep onset,
  sleep efficiency**,
- the **latency to each stage** (N1, N2, N3, REM),
- the **duration and percentage of each stage**, plus slow sleep (N2 + N3),
- the **event indices** per hour of sleep: apnea–hypopnea index (with its obstructive / central / mixed /
  hypopnea sub-indices), periodic limb movement index, arousal index,
- optionally the **oxygen desaturation index** (from the scored desaturations) and **T90** (time with
  SpO2 below 90 %, from the SpO2 signal).

**Inputs** — the remapped 30-s hypnogram written by tool 3 (`{file_id}_Hypnogram_remapped.txt`, one
AASM label per line), the scored events harmonised by tool 4 (`config_param/event_remap.json`), and the
EDF **header** (start time, duration, channel names). The EEG signal is **never** loaded; the only signal
reads are the optional `Light` channel (lights-off/on detection) and the `SpO2` channel (T90).

This tool is the one exception to the *feature tools start from the clean epochs* rule: macrostructure
needs the **complete** scored hypnogram, not the epochs that survived artefact rejection.

**Lights-off / lights-on** may come from various sources. The tool resolves them, in
order, from a `{file_id}_Summary_Export.txt` Profusion export (`Luminosité` column), the EDF `Light`
channel, two columns of the participant table, and finally the start/end of the recording — with a
warning whenever the fallback is used. Every result carries its `lights_source`.

**Outputs** — one long-format `{file_id}_sleep_metrics.tsv` (+ a `_checks.tsv` with the consistency
checks) per recording under `derivatives/features_macrostructure/`, and the database-level
`global_sleep_metrics.tsv` (one row per participant), an Excel workbook and an HTML report under
`reports_features_macrostructure/`. A last section compares the results with an external reference table
(e.g. the values of the clinical reports).

Work through the sections in order: **1** pick the data folder and scan, **2** set the parameters,
**3** choose the participants, **4** run, **5** (optional) compare with a reference table.
The **metric glossary** just below defines every metric and gives indicative normal ranges.

In [ ]:
# =============================================================================
# Tool 10 - Sleep macrostructure: setup, constants and shared configuration
# =============================================================================
import os
import re
import io
import json
import difflib
import datetime
import unicodedata
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import mne
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ipyfilechooser import FileChooser

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')
matplotlib.rcParams['figure.max_open_warning'] = 0

try:
    import openpyxl                     # noqa: F401  (pandas needs it to read/write .xlsx)
    HAS_OPENPYXL = True
except Exception:
    HAS_OPENPYXL = False

# --- sleep stages ------------------------------------------------------------
AASM_STAGES = ['W', 'N1', 'N2', 'N3', 'R']
SLEEP_STAGES = ['N1', 'N2', 'N3', 'R']      # everything counted as sleep

# MT (movement time), custom stages and unknown labels are "other": they follow YASA's convention -
# counted in time in bed (and inside the sleep period when they fall there) but excluded from both
# total sleep time and WASO. Set to 'wake' to count them as wake instead (WASO when inside the
# sleep period); the choice is recorded in the outputs.
OTHER_STAGE_POLICY = 'exclude'
# OTHER_STAGE_POLICY = 'wake'

# Window over which the scored events are counted for the per-hour indices: 'tib' = every event
# between lights-off and lights-on (denominator TST, the clinical convention); 'sleep_epochs' =
# only events whose onset falls in a sleep epoch.
EVENT_COUNT_WINDOW = 'tib'
# EVENT_COUNT_WINDOW = 'sleep_epochs'

DEFAULT_EPOCH_S = 30

# Output folder names (data vs reports split, toolkit convention)
DATA_DIRNAME = 'features_macrostructure'
REPORTS_DIRNAME = 'reports_features_macrostructure'

# Pipeline cost units for the per-participant progress bar
COST_HEADER, COST_HYPNO, COST_LIGHTS, COST_EVENTS, COST_SPO2, COST_WRITE = 2, 3, 20, 10, 20, 5

# --- metric registry ----------------------------------------------------------
# (key, label, unit, group, one-line definition) - the single source for the long TSV, the
# glossary, the workbook and the report. Order = column order of the global table.
METRICS = [
    ('tib_min', 'Time in bed (TIB)', 'min', 'macrostructure',
     'Lights-off to lights-on: number of epochs in bed x epoch length.'),
    ('sol_min', 'Sleep onset latency (SOL)', 'min', 'macrostructure',
     'Lights-off to sleep onset (rule chosen in Section 2; default = first non-W epoch).'),
    ('spt_min', 'Sleep period time (SPT)', 'min', 'macrostructure',
     'Sleep onset to the last sleep epoch, both included.'),
    ('tst_min', 'Total sleep time (TST)', 'min', 'macrostructure',
     'Number of N1 + N2 + N3 + R epochs within the sleep period x epoch length.'),
    ('waso_min', 'Wake after sleep onset (WASO)', 'min', 'macrostructure',
     'Number of W epochs within the sleep period x epoch length.'),
    ('other_min', 'Other epochs within SPT', 'min', 'macrostructure',
     'MT / custom / unknown epochs within the sleep period (neither sleep nor wake; see policy).'),
    ('se_pct', 'Sleep efficiency (SE)', '%', 'macrostructure', '100 x TST / TIB.'),
    ('sme_pct', 'Sleep maintenance efficiency (SME)', '%', 'macrostructure', '100 x TST / SPT.'),
    ('lat_n1_min', 'Latency to N1', 'min', 'macrostructure',
     'Sleep onset to the first N1 epoch (0 under the default onset rule when onset is N1).'),
    ('lat_n2_min', 'Latency to N2', 'min', 'macrostructure', 'Sleep onset to the first N2 epoch.'),
    ('lat_n3_min', 'Latency to N3', 'min', 'macrostructure', 'Sleep onset to the first N3 epoch.'),
    ('lat_rem_min', 'REM latency', 'min', 'macrostructure', 'Sleep onset to the first R epoch.'),
    ('n1_min', 'N1 duration', 'min', 'macrostructure', 'Number of N1 epochs within SPT x epoch length.'),
    ('n2_min', 'N2 duration', 'min', 'macrostructure', 'Number of N2 epochs within SPT x epoch length.'),
    ('n3_min', 'N3 duration', 'min', 'macrostructure', 'Number of N3 epochs within SPT x epoch length.'),
    ('rem_min', 'REM duration', 'min', 'macrostructure', 'Number of R epochs within SPT x epoch length.'),
    ('n1_pct', 'N1 (% of TST)', '%', 'macrostructure', '100 x N1 duration / TST.'),
    ('n2_pct', 'N2 (% of TST)', '%', 'macrostructure', '100 x N2 duration / TST.'),
    ('n3_pct', 'N3 (% of TST)', '%', 'macrostructure', '100 x N3 duration / TST.'),
    ('rem_pct', 'REM (% of TST)', '%', 'macrostructure', '100 x REM duration / TST.'),
    ('slow_sleep_min', 'Slow sleep (N2 + N3) duration', 'min', 'macrostructure', 'N2 + N3 duration.'),
    ('slow_sleep_pct', 'Slow sleep (N2 + N3, % of TST)', '%', 'macrostructure', 'N2 % + N3 %.'),
    ('n_apnea_obstructive', 'Obstructive apneas', 'count', 'respiratory',
     'Scored obstructive apneas within the counting window.'),
    ('n_apnea_central', 'Central apneas', 'count', 'respiratory',
     'Scored central apneas within the counting window.'),
    ('n_apnea_mixed', 'Mixed apneas', 'count', 'respiratory',
     'Scored mixed apneas within the counting window.'),
    ('n_hypopnea', 'Hypopneas', 'count', 'respiratory',
     'Scored hypopneas (any canonical label starting with "hypopnea") within the counting window.'),
    ('oai', 'Obstructive apnea index (OAI)', 'events/h', 'respiratory', 'Obstructive apneas / TST (h).'),
    ('cai', 'Central apnea index (CAI)', 'events/h', 'respiratory', 'Central apneas / TST (h).'),
    ('mai', 'Mixed apnea index (MAI)', 'events/h', 'respiratory', 'Mixed apneas / TST (h).'),
    ('hi', 'Hypopnea index (HI)', 'events/h', 'respiratory', 'Hypopneas / TST (h).'),
    ('ahi', 'Apnea-hypopnea index (AHI)', 'events/h', 'respiratory',
     '(all apneas + hypopneas) / TST (h) = OAI + CAI + MAI + HI.'),
    ('n_plm', 'Periodic limb movements', 'count', 'plm',
     'Scored PLM events (canonical "plm") within the counting window.'),
    ('plm_index', 'PLM index', 'events/h', 'plm', 'PLM events / TST (h).'),
    ('n_arousal', 'Arousals', 'count', 'arousal',
     'Scored arousals (any canonical label starting with "arousal") within the counting window.'),
    ('arousal_index', 'Arousal index', 'events/h', 'arousal', 'Arousals / TST (h).'),
    ('n_desaturation', 'SpO2 desaturations', 'count', 'odi',
     'Scored desaturations within the counting window (depth >= threshold when the depth is known).'),
    ('odi', 'Oxygen desaturation index (ODI)', 'events/h', 'odi', 'Desaturations / TST (h).'),
    ('t90_min', 'T90', 'min', 'spo2', 'Time with SpO2 < threshold (default 90 %) between lights-off and lights-on.'),
    ('t90_pct_tst', 'T90 (% of TST)', '%', 'spo2', '100 x T90 / TST.'),
    ('spo2_mean_pct', 'Mean SpO2', '%', 'spo2', 'Mean of the valid SpO2 samples in bed.'),
    ('spo2_nadir_pct', 'SpO2 nadir', '%', 'spo2', 'Minimum of the valid SpO2 samples in bed.'),
    ('spo2_artifact_pct', 'SpO2 artefact', '%', 'spo2',
     'Percentage of SpO2 samples in bed discarded as artefact (<= 0 or below the floor).'),
]
METRIC_KEYS = [m[0] for m in METRICS]
METRIC_INFO = {m[0]: {'label': m[1], 'unit': m[2], 'group': m[3], 'definition': m[4]} for m in METRICS}

# Provenance written with every participant (text columns of the global table)
PROVENANCE_KEYS = [
    'lights_source', 'lights_off_clock', 'lights_on_clock', 'lights_off_epoch', 'lights_on_epoch',
    'sleep_onset_rule', 'sleep_onset_epoch', 'other_stage_policy', 'event_count_window',
    'epoch_length_s', 'n_epochs_hypno', 'n_epochs_in_bed', 'n_mt_custom', 'event_source',
    'n_events_total', 'n_unmapped_events', 'n_events_outside_window', 'has_spo2', 'odi_threshold_pct',
    'odi_rule', 'spo2_channel', 'light_channel', 'recording_start', 'edf_duration_min', 'n_warnings']

# --- indicative normal ranges (healthy adults) --------------------------------
# INDICATIVE ONLY: these values are textbook / consensus ranges for healthy adults recorded in a
# sleep laboratory; they depend on age (N3 and SE fall, WASO and N1 rise with age), on the
# first-night effect and on the scoring rules. Edit freely; they are only used to shade the
# figures of the database report and to annotate the glossary. (low, high, note)
REFERENCE_RANGES = {
    'sol_min':        (0, 30,   'typically 10-20 min; > 30 min suggests difficulty initiating sleep'),
    'waso_min':       (0, 30,   'rises with age; > 30-40 min suggests sleep maintenance difficulty'),
    'se_pct':         (85, 100, 'often lower on a first laboratory night'),
    'sme_pct':        (90, 100, 'sleep continuity once asleep'),
    'lat_rem_min':    (60, 120, '< 15 min = sleep-onset REM period (narcolepsy, REM rebound, depression); prolonged by REM-suppressing medication'),
    'n1_pct':         (2, 5,    'up to ~10 % in older adults; high N1 = fragmented sleep'),
    'n2_pct':         (45, 55,  ''),
    'n3_pct':         (13, 23,  'falls with age; depends on the amplitude criterion of the scorer'),
    'rem_pct':        (20, 25,  ''),
    'slow_sleep_pct': (58, 78,  'N2 + N3'),
    'ahi':            (0, 5,    'AASM severity: < 5 normal, 5-15 mild, 15-30 moderate, > 30 severe'),
    'plm_index':      (0, 15,   'PLMD threshold: > 15/h in adults (> 5/h in children)'),
    'arousal_index':  (0, 15,   '~10/h in young adults, higher with age; > 15-20/h = fragmented sleep'),
    'odi':            (0, 5,    'same severity bands as the AHI are often used; state the threshold (3 % or 4 %)'),
    't90_pct_tst':    (0, 10,   '> 10 % of the night below 90 % = clinically significant nocturnal hypoxaemia'),
    'spo2_mean_pct':  (94, 100, ''),
    'spo2_nadir_pct': (88, 100, 'a nadir below 88 % is generally considered clinically relevant'),
}
AHI_SEVERITY_BANDS = [(0, 5, 'normal'), (5, 15, 'mild'), (15, 30, 'moderate'), (30, None, 'severe')]


def norm(p):
    """Normalised path/id string, used on BOTH sides of every comparison (Windows safety)."""
    return os.path.normcase(str(p))


def strip_accents(text):
    """'Luminosité' -> 'Luminosite' (accent-insensitive column matching)."""
    return ''.join(c for c in unicodedata.normalize('NFKD', str(text)) if not unicodedata.combining(c))


def parse_custom_field(text):
    """Parse the comma-separated 'Custom stages' field into a clean, de-duplicated list."""
    seen = []
    for tok in str(text).split(','):
        tok = tok.strip()
        if tok and tok not in seen:
            seen.append(tok)
    return seen


def load_custom_stages(folder):
    """Read config_param/custom_stages.json under the data folder ([] when absent)."""
    if not folder:
        return []
    path = Path(folder) / 'config_param' / 'custom_stages.json'
    if not path.exists():
        return []
    try:
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        stages = data.get('custom_stages', []) if isinstance(data, dict) else []
        return [str(s) for s in stages]
    except Exception:
        return []


def load_subject_info(path):
    """Read the optional participant-info table (.csv/.tsv/.xlsx). Returns (DataFrame, error)."""
    if not path:
        return None, ''
    try:
        if norm(path).endswith(('.xlsx', '.xls')):
            if not HAS_OPENPYXL:
                return None, 'openpyxl is not installed - save the table as .csv or .tsv'
            df = pd.read_excel(path)
        else:
            sep = '\t' if norm(path).endswith('.tsv') else None
            df = pd.read_csv(path, sep=sep, engine='python')
        return df, ''
    except Exception as exc:
        return None, str(exc)

## Metric glossary

All metrics are computed on the epochs between **lights-off** and **lights-on** (epoch length 30 s by
default, `ep` = epoch length in minutes). Epoch indices are 0-based; lights-on is the first epoch *out*
of bed. "Sleep" = N1, N2, N3, R. The indicative ranges are textbook values for **healthy adults** — they
shift with age, medication and the first-night effect, and are only meant as a quick sanity check
(they can be edited in the `REFERENCE_RANGES` dictionary above).

| Metric | Definition / formula | Typical use | Indicative range (healthy adults) |
|---|---|---|---|
| **TIB** — time in bed | lights-off → lights-on: `n epochs in bed × ep` | Denominator of sleep efficiency | — |
| **Sleep onset** | first epoch after lights-off scored N1/N2/N3/R (default rule). Alternatives: first N2 epoch; first of 3 consecutive sleep epochs | Anchor for SOL, SPT and every latency | — |
| **SOL** — sleep onset latency | lights-off → sleep onset: `onset index × ep` | Sleep initiation (insomnia, hypersomnia — short SOL on an MSLT) | 10–20 min; > 30 min prolonged |
| **SPT** — sleep period time | sleep onset → last sleep epoch (both included) | Window of the sleep episode | — |
| **TST** — total sleep time | `(n N1 + n N2 + n N3 + n R within SPT) × ep` | Denominator of every per-hour index and stage % | 6–8 h at home, often less in the lab |
| **WASO** — wake after sleep onset | `n W epochs within SPT × ep` | Sleep maintenance | < 30 min; rises with age |
| **Other within SPT** | MT / custom / unknown epochs within SPT | Reported so that `SPT = TST + WASO + other` | 0 |
| **SE** — sleep efficiency | `100 × TST / TIB` | Global sleep quality; the insomnia criterion | > 85 % |
| **SME** — sleep maintenance efficiency | `100 × TST / SPT` | Continuity once asleep (independent of SOL) | > 90 % |
| **Latency to N1 / N2 / N3 / REM** | sleep onset → first epoch of the stage; undefined (NaN) if the stage never occurs | REM latency: narcolepsy (< 15 min = SOREMP), depression, medication | REM: 60–120 min |
| **Stage duration** | `n epochs of the stage within SPT × ep` | Absolute amount of each stage | — |
| **Stage % of TST** | `100 × stage duration / TST` | Sleep architecture; sums to 100 % | N1 2–5 %, N2 45–55 %, N3 13–23 %, REM 20–25 % |
| **Slow sleep (N2 + N3)** | N2 + N3 duration, N2 % + N3 % | Slow-wave-oriented analyses | 58–78 % |
| **AHI** and **OAI / CAI / MAI / HI** | `n events / TST (h)`; AHI = OAI + CAI + MAI + HI | Sleep apnea diagnosis and severity | < 5 normal; 5–15 mild; 15–30 moderate; > 30 severe |
| **PLM index** | `n periodic limb movements / TST (h)` | Periodic limb movement disorder | < 15/h (adults) |
| **Arousal index** | `n arousals / TST (h)` | Sleep fragmentation | ~10/h young adults, higher with age |
| **ODI** — oxygen desaturation index | `n desaturations (≥ 3 % or 4 %) / TST (h)` | Hypoxic burden of respiratory events; state the threshold | < 5/h |
| **T90** | time with SpO2 < 90 % in bed, in min and `% of TST` | Nocturnal hypoxaemia | < 10 % of TST |

**Consistency checks** run on every recording (written to `{file_id}_sleep_metrics_checks.tsv` and
summarised in the report): `SPT = TST + WASO + other`, `0 ≤ SE ≤ 100`, `TST ≤ SPT ≤ TIB`,
`SOL + SPT ≤ TIB`, stage % sum to 100, latencies within `[0, SPT]`, hypnogram length vs EDF duration,
lights file length vs hypnogram, lights-off < lights-on, event onsets inside the recording, unmapped
event labels, MT / custom stages present, no sleep at all.

**MT and custom stages** (YASA convention): counted in TIB and in SPT, excluded from TST and WASO. The
tool warns when it meets them; if you prefer them scored as a neighbouring stage, remap them with tool 3
(or set `OTHER_STAGE_POLICY = 'wake'` above to count them as wake).

**Lights-off / lights-on detection** (`summary txt` and `Light channel` sources): lights-off = the first
epoch after the **last lights-ON epoch that precedes the first sleep epoch**; lights-on = the **first
lights-ON epoch after the last sleep epoch**. Without an ON epoch on either side the recording start /
end is used instead (warning). ON epochs inside the sleep period are counted and reported, not used.

In [ ]:
# =============================================================================
# Core computation - EDF header, hypnogram, lights, macrostructure, events, SpO2, checks
# (pure functions: no widget is touched here)
# =============================================================================

# ---------------------------------------------------------------------------
# EDF header (header-only binary read - the signal is never loaded here)
# ---------------------------------------------------------------------------
def read_edf_header_info(edf_path):
    """Read the fixed EDF header and the per-channel labels without touching the signal.

    Same byte offsets as tool 1's custom parser: 168 start date 'dd.mm.yy', 176 start time
    'hh.mm.ss', 184 header size, 236 number of data records, 244 record duration (s), 252 number
    of channels, then 16 bytes x n labels ... 8 bytes x n samples-per-record. The recording
    duration is `n_records x record_duration`; when the record count is -1 (streaming export) it
    is recovered from the file size. edfio is NOT used on purpose: it fails on files whose EDF+
    annotation record is empty (tools/test_data/73.edf). Raises on any failure (fatal)."""
    edf_path = Path(edf_path)
    with open(edf_path, 'rb') as f:
        head = f.read(256)
        if len(head) < 256:
            raise ValueError('file shorter than the 256-byte EDF header')
        date_str = head[168:176].decode('ascii', 'replace').strip()
        time_str = head[176:184].decode('ascii', 'replace').strip()
        header_bytes = int(head[184:192].decode('ascii', 'replace').strip())
        n_records = int(head[236:244].decode('ascii', 'replace').strip())
        rec_dur = float(head[244:252].decode('ascii', 'replace').strip())
        n_ch = int(head[252:256].decode('ascii', 'replace').strip())
        block = f.read(n_ch * 256)
    dd, mm, yy = (int(x) for x in date_str.split('.'))
    hh, mi, ss = (int(x) for x in time_str.split('.'))
    year = 2000 + yy if yy <= 84 else 1900 + yy          # EDF 2-digit-year rule
    start_dt = datetime.datetime(year, mm, dd, hh, mi, ss)

    def field(offset, width):
        return [block[offset + i * width: offset + (i + 1) * width].decode('latin-1').strip()
                for i in range(n_ch)]
    labels = field(0, 16)
    dims = field(16 * n_ch + 80 * n_ch, 8)
    spr_off = 16 * n_ch + 80 * n_ch + 8 * n_ch + 8 * n_ch * 4 + 80 * n_ch
    samples_per_record = [int(float(x)) if x else 0 for x in field(spr_off, 8)]
    if n_records < 0:                                     # unknown length: infer from the size
        bytes_per_record = 2 * sum(samples_per_record)
        n_records = (edf_path.stat().st_size - header_bytes) // max(bytes_per_record, 1)
    duration_s = n_records * rec_dur
    sfreq = {lab: (spr / rec_dur if rec_dur > 0 else np.nan)
             for lab, spr in zip(labels, samples_per_record)}
    return {'start_dt': start_dt, 'duration_s': float(duration_s), 'n_data_records': int(n_records),
            'record_duration_s': rec_dur, 'n_channels': n_ch, 'ch_names': labels,
            'samples_per_record': samples_per_record, 'sfreq_per_ch': sfreq, 'dims': dims}


def find_channel(ch_names, pattern):
    """First channel label matching a case-insensitive regex, or None."""
    rx = re.compile(pattern, re.IGNORECASE)
    for ch in ch_names:
        if rx.search(ch):
            return ch
    return None


LIGHT_CH_PATTERN = r'^\s*lights?\s*$|lumin'
SPO2_CH_PATTERN = r'spo2|sao2'


# ---------------------------------------------------------------------------
# Hypnogram
# ---------------------------------------------------------------------------
def load_hypnogram(path):
    """One label per line (tool 3 output) -> array of str."""
    return np.loadtxt(str(path), dtype=str).astype('<U10')


def reconcile_hypno_length(stages, duration_s, epoch_s):
    """Compare the hypnogram with the EDF duration. One extra scored epoch (the recording ends
    mid-epoch) is dropped; one missing epoch is kept as is; anything else is a mismatch.
    Returns (stages, check) with check['status'] in {'ok', 'warning', 'fail'}."""
    expected = int(np.floor(duration_s / epoch_s))
    n = len(stages)
    if n == expected:
        return stages, {'check': 'hypno_vs_edf_length', 'status': 'ok',
                        'detail': f'{n} epochs in both'}
    if n - expected == 1:
        return stages[:expected], {'check': 'hypno_vs_edf_length', 'status': 'warning',
                                   'detail': f'hypnogram has 1 epoch more than the EDF ({n} vs '
                                             f'{expected}) - last epoch dropped'}
    if expected - n == 1:
        return stages, {'check': 'hypno_vs_edf_length', 'status': 'warning',
                        'detail': f'hypnogram has 1 epoch less than the EDF ({n} vs {expected}) - kept'}
    return stages, {'check': 'hypno_vs_edf_length', 'status': 'fail',
                    'detail': f'hypnogram has {n} epochs, the EDF {expected} - mismatch > 1 epoch'}


def classify_stages(stages, custom_stages):
    """Boolean masks over the hypnogram: sleep (N1/N2/N3/R), wake (W) and "other" (MT, custom
    stages, unknown labels). Under OTHER_STAGE_POLICY = 'wake' the other epochs join the wake
    mask instead (YASA-style exclusion is the default)."""
    stages = np.asarray(stages, dtype=str)
    sleep = np.isin(stages, SLEEP_STAGES)
    wake = stages == 'W'
    other = ~(sleep | wake)
    known = set(AASM_STAGES) | {'MT'} | set(custom_stages)
    unknown = sorted(str(x) for x in set(stages[other]) - known)
    if OTHER_STAGE_POLICY == 'wake':
        wake = wake | other
        other = np.zeros_like(other)
    return {'sleep_mask': sleep, 'wake_mask': wake, 'other_mask': other, 'unknown_labels': unknown}


# ---------------------------------------------------------------------------
# Lights-off / lights-on
# ---------------------------------------------------------------------------
def load_lights_from_summary_txt(path, col='Luminosité'):
    """Read the per-epoch light state from a Profusion *_Summary_Export.txt (comma-separated,
    French header, UTF-16 with BOM or UTF-8). Column `col` (accent/case-insensitive match):
    1 = lights ON, 0 = lights OFF. Returns a boolean array (True = ON). Raises when the column
    is missing (the caller falls through to the next source)."""
    raw = open(path, 'rb').read()
    if raw[:2] in (b'\xff\xfe', b'\xfe\xff'):
        text = raw.decode('utf-16')
    else:
        text = raw.decode('utf-8-sig', errors='replace')
    df = pd.read_csv(io.StringIO(text), sep=',')
    target = strip_accents(col).strip().lower()
    match = [c for c in df.columns if strip_accents(c).strip().lower() == target]
    if not match:
        raise ValueError(f'column "{col}" not found (columns: {", ".join(map(str, df.columns))})')
    values = pd.to_numeric(df[match[0]], errors='coerce').fillna(0).to_numpy()
    return values == 1


def detect_lights_from_channel(edf_path, ch_name, epoch_s):
    """Per-epoch light state from an EDF step channel (Compumedics 'Light': 1 = ON, 0 = OFF).
    Only that channel is read (`include=` at read time). Returns a boolean array (True = ON)."""
    raw = mne.io.read_raw_edf(str(edf_path), include=[ch_name], preload=True, verbose=False)
    x = raw.get_data()[0]
    sf = raw.info['sfreq']
    n_per_epoch = int(round(epoch_s * sf))
    n = len(x) // n_per_epoch
    if n == 0:
        raise ValueError('Light channel shorter than one epoch')
    med = np.median(x[:n * n_per_epoch].reshape(n, n_per_epoch), axis=1)
    return med > 0.5


def align_lights_to_hypno(on, n_hyp):
    """Bring a per-epoch light array to the hypnogram length: truncate a longer one, pad a
    one-epoch-short one with its last value, reject a larger mismatch (returns None)."""
    on = np.asarray(on, dtype=bool)
    n = len(on)
    if n == n_hyp:
        return on, None
    if n > n_hyp:
        return on[:n_hyp], {'check': 'lights_txt_vs_hypno_length', 'status': 'warning',
                            'detail': f'light file has {n} epochs, hypnogram {n_hyp} - truncated'}
    if n_hyp - n == 1:
        return np.concatenate([on, on[-1:]]), {
            'check': 'lights_txt_vs_hypno_length', 'status': 'warning',
            'detail': f'light file has {n} epochs, hypnogram {n_hyp} - last value repeated'}
    return None, {'check': 'lights_txt_vs_hypno_length', 'status': 'fail',
                  'detail': f'light file has {n} epochs, hypnogram {n_hyp} - source rejected'}


def lights_from_on_flags(on, sleep_mask):
    """Lights-off / lights-on epoch indices from a per-epoch ON/OFF array.

    lights-off = the first epoch AFTER the last ON epoch that precedes the first sleep epoch
    (a bare "first OFF epoch" rule fails when the recording starts with the light already off
    and the technician switches it on briefly before the night). lights-on = the first ON epoch
    after the last sleep epoch (exclusive index). Without an ON epoch on a side the recording
    start / end is used, with a warning. ON epochs inside the sleep period are reported, not used.
    Returns (off_idx, on_idx, checks)."""
    on = np.asarray(on, dtype=bool)
    sleep = np.asarray(sleep_mask, dtype=bool)
    n = len(on)
    checks = []
    if not sleep.any():
        before = np.where(on)[0]
        off_idx = int(before[-1] + 1) if len(before) and before[-1] + 1 < n else 0
        checks.append({'check': 'no_lights_on', 'status': 'warning',
                       'detail': 'no sleep epoch - lights-on set to the recording end'})
        return off_idx, n, checks
    first_sleep = int(np.argmax(sleep))
    last_sleep = int(n - 1 - np.argmax(sleep[::-1]))
    before = np.where(on[:first_sleep])[0]
    if len(before):
        off_idx = int(before[-1] + 1)
    else:
        off_idx = 0
        checks.append({'check': 'no_lights_off', 'status': 'warning',
                       'detail': 'no lights-ON epoch before the first sleep epoch - lights-off '
                                 'set to the recording start'})
    after = np.where(on[last_sleep + 1:])[0]
    if len(after):
        on_idx = int(last_sleep + 1 + after[0])
    else:
        on_idx = n
        checks.append({'check': 'no_lights_on', 'status': 'warning',
                       'detail': 'no lights-ON epoch after the last sleep epoch - lights-on set '
                                 'to the recording end'})
    n_inside = int(on[first_sleep:last_sleep + 1].sum())
    if n_inside:
        checks.append({'check': 'lights_on_inside_sleep', 'status': 'warning',
                       'detail': f'{n_inside} lights-ON epoch(s) between the first and last sleep '
                                 f'epoch - ignored (bathroom break, technician?)'})
    return off_idx, on_idx, checks


def _parse_lights_cell(value, start_dt, epoch_s):
    """One participant-table cell -> epoch index. Accepts 'HH:MM', 'HH:MM:SS' (clock time on the
    recording date, rolled to the next day when it precedes the start by more than 1 h) or a
    bare integer (epoch index). Returns None for a blank/unparseable cell."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).strip()
    if not text or text.lower() in ('nan', 'none', 'nat'):
        return None
    if re.fullmatch(r'\d+', text):
        return int(text)
    m = re.fullmatch(r'(\d{1,2})[:h.](\d{2})(?::(\d{2}))?', text)
    if m is None:
        try:                                           # a pandas/Excel time object
            t = pd.to_datetime(text).time()
        except Exception:
            return None
    else:
        t = datetime.time(int(m.group(1)), int(m.group(2)), int(m.group(3) or 0))
    dt = datetime.datetime.combine(start_dt.date(), t)
    if (start_dt - dt).total_seconds() > 3600:         # after midnight -> next calendar day
        dt += datetime.timedelta(days=1)
    return int(round((dt - start_dt).total_seconds() / epoch_s))


def lights_from_participant_table(row, start_dt, epoch_s, n_hyp, off_col, on_col):
    """Lights-off / lights-on from two columns of the participant table (clock time or epoch
    index). A blank cell leaves that side to the recording bound. Returns (off, on, checks) or
    (None, None, checks) when neither cell is usable."""
    checks = []
    off = _parse_lights_cell(row.get(off_col) if row is not None else None, start_dt, epoch_s)
    on = _parse_lights_cell(row.get(on_col) if row is not None else None, start_dt, epoch_s)
    if off is None and on is None:
        return None, None, checks
    if off is None:
        off = 0
        checks.append({'check': 'no_lights_off', 'status': 'warning',
                       'detail': f'participant table: no "{off_col}" value - recording start used'})
    if on is None:
        on = n_hyp
        checks.append({'check': 'no_lights_on', 'status': 'warning',
                       'detail': f'participant table: no "{on_col}" value - recording end used'})
    if off < 0 or off > n_hyp or on < 0 or on > n_hyp:
        checks.append({'check': 'lights_order', 'status': 'warning',
                       'detail': f'participant table lights outside the recording (off={off}, '
                                 f'on={on}, n={n_hyp}) - clamped'})
        off, on = int(np.clip(off, 0, n_hyp)), int(np.clip(on, 0, n_hyp))
    return off, on, checks


def resolve_lights(cfg, p, header, n_hyp, sleep_mask, subj_row):
    """Try the lights sources in the configured order and return
    {'off_idx', 'on_idx', 'source', 'light_channel', 'checks'}.

    Sources: 'summary_txt' (Profusion export next to the EDF), 'light_channel' (EDF step
    channel), 'participant_table' (two columns), 'recording_bounds' (fallback, always warned).
    'auto' walks them in that order; a forced source that is unavailable falls back to the
    recording bounds with a warning naming what was missing."""
    order = (['summary_txt', 'light_channel', 'participant_table'] if cfg['lights_source'] == 'auto'
             else [cfg['lights_source']])
    checks = []
    light_ch = p.get('light_ch')
    for src in order:
        try:
            if src == 'summary_txt':
                path = p.get('summary_txt')
                if path is None:
                    checks.append({'check': 'lights_source', 'status': 'info',
                                   'detail': 'no summary txt next to the EDF'})
                    continue
                on = load_lights_from_summary_txt(path, cfg['lights_col'])
                on, chk = align_lights_to_hypno(on, n_hyp)
                if chk is not None:
                    checks.append(chk)
                if on is None:
                    continue
                off_idx, on_idx, c2 = lights_from_on_flags(on, sleep_mask)
                return {'off_idx': off_idx, 'on_idx': on_idx, 'source': 'summary_txt',
                        'light_channel': '', 'checks': checks + c2}
            if src == 'light_channel':
                if not light_ch:
                    checks.append({'check': 'lights_source', 'status': 'info',
                                   'detail': 'no Light channel in the EDF'})
                    continue
                on = detect_lights_from_channel(p['edf'], light_ch, cfg['epoch_s'])
                on, chk = align_lights_to_hypno(on, n_hyp)
                if chk is not None:
                    checks.append(chk)
                if on is None:
                    continue
                off_idx, on_idx, c2 = lights_from_on_flags(on, sleep_mask)
                return {'off_idx': off_idx, 'on_idx': on_idx, 'source': 'light_channel',
                        'light_channel': light_ch, 'checks': checks + c2}
            if src == 'participant_table':
                off_idx, on_idx, c2 = lights_from_participant_table(
                    subj_row, header['start_dt'], cfg['epoch_s'], n_hyp,
                    cfg['lights_off_col'], cfg['lights_on_col'])
                if off_idx is None:
                    checks.append({'check': 'lights_source', 'status': 'info',
                                   'detail': 'no lights columns / values in the participant table'})
                    continue
                return {'off_idx': off_idx, 'on_idx': on_idx, 'source': 'participant_table',
                        'light_channel': '', 'checks': checks + c2}
            if src == 'recording_bounds':
                break
        except Exception as exc:
            checks.append({'check': 'lights_source', 'status': 'warning',
                           'detail': f'{src} unusable ({exc}) - next source tried'})
    tried = ', '.join(order) if cfg['lights_source'] == 'auto' else cfg['lights_source']
    if cfg['lights_source'] != 'recording_bounds':
        checks.append({'check': 'lights_fallback', 'status': 'warning',
                       'detail': f'no usable lights source ({tried}) - the recording start and end '
                                 f'are used as lights-off / lights-on'})
    else:
        checks.append({'check': 'lights_fallback', 'status': 'info',
                       'detail': 'recording start / end used as lights-off / lights-on (as requested)'})
    return {'off_idx': 0, 'on_idx': n_hyp, 'source': 'recording_bounds', 'light_channel': '',
            'checks': checks}


# ---------------------------------------------------------------------------
# Macrostructure
# ---------------------------------------------------------------------------
ONSET_RULES = ['first non-W epoch', 'first N2 epoch', 'first of 3 consecutive sleep epochs']


def find_sleep_onset(seg, sleep_seg, rule):
    """Index (within the in-bed segment) of the sleep onset under `rule`, or None."""
    seg = np.asarray(seg, dtype=str)
    sleep_seg = np.asarray(sleep_seg, dtype=bool)
    if rule == 'first N2 epoch':
        hits = np.where(seg == 'N2')[0]
        return int(hits[0]) if len(hits) else None
    if rule == 'first of 3 consecutive sleep epochs':
        for i in range(len(sleep_seg) - 2):
            if sleep_seg[i] and sleep_seg[i + 1] and sleep_seg[i + 2]:
                return i
        return None
    hits = np.where(sleep_seg)[0]                       # default: first non-W (= first sleep) epoch
    return int(hits[0]) if len(hits) else None


def compute_macrostructure(stages, cls, off_idx, on_idx, epoch_s, onset_rule):
    """Every hypnogram-based metric on the epochs [off_idx, on_idx).

    Indices are 0-based, lights-on exclusive. Stage durations are counted WITHIN the sleep
    period (identical to the whole-night count under the default onset rule; under the N2 /
    3-consecutive rules the sleep epochs before onset are left out and reported).
    No sleep at all -> TIB only, everything else NaN.
    Returns (metrics dict, checks list, extras dict)."""
    ep = epoch_s / 60.0
    stages = np.asarray(stages, dtype=str)
    seg = stages[off_idx:on_idx]
    sleep = cls['sleep_mask'][off_idx:on_idx]
    wake = cls['wake_mask'][off_idx:on_idx]
    other = cls['other_mask'][off_idx:on_idx]
    n = len(seg)
    metrics = {k: np.nan for k, _, _, g, _ in METRICS if g == 'macrostructure'}
    metrics['tib_min'] = n * ep
    checks = []
    extras = {'onset_idx_abs': None, 'last_sleep_idx_abs': None, 'n_epochs_in_bed': n,
              'n_other_in_bed': int(other.sum()), 'n_sleep_before_onset': 0}
    onset = find_sleep_onset(seg, sleep, onset_rule)
    if onset is None or not sleep.any():
        checks.append({'check': 'no_sleep', 'status': 'fail',
                       'detail': 'no sleep onset found between lights-off and lights-on - every '
                                 'sleep metric is NaN'})
        return metrics, checks, extras
    last = int(n - 1 - np.argmax(sleep[::-1]))
    spt_seg, spt_sleep, spt_wake, spt_other = (seg[onset:last + 1], sleep[onset:last + 1],
                                               wake[onset:last + 1], other[onset:last + 1])
    metrics['sol_min'] = onset * ep
    metrics['spt_min'] = (last - onset + 1) * ep
    metrics['tst_min'] = int(spt_sleep.sum()) * ep
    metrics['waso_min'] = int(spt_wake.sum()) * ep
    metrics['other_min'] = int(spt_other.sum()) * ep
    metrics['se_pct'] = 100.0 * metrics['tst_min'] / metrics['tib_min'] if metrics['tib_min'] > 0 else np.nan
    metrics['sme_pct'] = 100.0 * metrics['tst_min'] / metrics['spt_min'] if metrics['spt_min'] > 0 else np.nan
    for stage, key in [('N1', 'n1'), ('N2', 'n2'), ('N3', 'n3'), ('R', 'rem')]:
        hits = np.where(spt_seg == stage)[0]
        metrics[f'lat_{key}_min'] = hits[0] * ep if len(hits) else np.nan
        metrics[f'{key}_min'] = len(hits) * ep
        metrics[f'{key}_pct'] = (100.0 * metrics[f'{key}_min'] / metrics['tst_min']
                                 if metrics['tst_min'] > 0 else np.nan)
    metrics['slow_sleep_min'] = metrics['n2_min'] + metrics['n3_min']
    metrics['slow_sleep_pct'] = metrics['n2_pct'] + metrics['n3_pct']
    extras['onset_idx_abs'] = off_idx + onset
    extras['last_sleep_idx_abs'] = off_idx + last
    extras['n_sleep_before_onset'] = int(sleep[:onset].sum())
    if extras['n_sleep_before_onset']:
        checks.append({'check': 'sleep_before_onset', 'status': 'info',
                       'detail': f"{extras['n_sleep_before_onset']} sleep epoch(s) before the "
                                 f'"{onset_rule}" onset are not counted in TST / stage durations'})
    return metrics, checks, extras


_TXT_DUR_RE = re.compile(r'^(\d+):(\d+(?:\.\d+)?)$')   # duration column "M:SS" or "M:SS.s"


def read_edf_start_datetime(edf_path):
    """Read the EDF recording-start datetime from the fixed header (offset 168 = date
    'dd.mm.yy', 176 = time 'hh.mm.ss'), applying the EDF 2-digit-year clipping
    (00-84 -> 20xx, 85-99 -> 19xx). Header-only read; returns a datetime or None on failure."""
    try:
        with open(edf_path, 'rb') as f:
            f.seek(168)
            date_str = f.read(8).decode('ascii', 'replace').strip()   # dd.mm.yy
            time_str = f.read(8).decode('ascii', 'replace').strip()   # hh.mm.ss
        dd, mm, yy = (int(x) for x in date_str.split('.'))
        hh, mi, ss = (int(x) for x in time_str.split('.'))
        year = 2000 + yy if yy <= 84 else 1900 + yy
        return datetime.datetime(year, mm, dd, hh, mi, ss)
    except Exception:
        return None


def event_companion_paths(edf_path, txt_suffix='_ScoredEvents_Export.txt', csv_suffix='_event_xml.csv'):
    """Return (txt_path_or_None, csv_path_or_None, xml_path_or_None) for an EDF stem."""
    edf_path = Path(edf_path)
    txt = edf_path.with_name(f'{edf_path.stem}{txt_suffix}')
    txt = txt if txt.exists() else None
    csv = edf_path.with_name(f'{edf_path.stem}{csv_suffix}')
    csv = csv if csv.exists() else None
    xml = None
    for cand in (f'{edf_path.name}.XML', f'{edf_path.name}.xml'):
        p = edf_path.with_name(cand)
        if p.exists():
            xml = p
            break
    return txt, csv, xml


def _events_df_from_txt(txt_path, rec_start):
    """Parse a Compumedics/Curry French text export (*_ScoredEvents_Export.txt)
    -> DataFrame Name/Start/Duration (seconds), or None. Comma-separated, no header, columns:
        HH:MM:SS , epoch# , stage_FR , event_label_FR , M:SS[.s] , - , - , position
    Encoding varies (UTF-16 with BOM on some exports, UTF-8/ANSI on others) -> the BOM is sniffed.
    Clock times are converted to seconds-from-recording-start (with midnight rollover) using rec_start."""
    raw = open(txt_path, 'rb').read()
    if raw[:2] in (b'\xff\xfe', b'\xfe\xff'):
        text = raw.decode('utf-16')
    else:
        text = raw.decode('utf-8', errors='replace')
    rec_date = rec_start.date()
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = [p.strip() for p in line.split(',')]
        if len(parts) < 5:
            continue
        name = parts[3]
        try:
            hh, mm, ss = parts[0].split(':')   # hours may be single-digit
            clock_t = datetime.time(int(hh), int(mm), int(ss))
        except (ValueError, TypeError):
            continue
        event_dt = datetime.datetime.combine(rec_date, clock_t)
        # events recorded after midnight fall on the next calendar day
        if (rec_start - event_dt).total_seconds() > 3600:
            event_dt += datetime.timedelta(days=1)
        start_sec = (event_dt - rec_start).total_seconds()
        m = _TXT_DUR_RE.match(parts[4])
        dur_sec = int(m.group(1)) * 60 + float(m.group(2)) if m else 0.0
        rows.append((name, start_sec, dur_sec))
    if not rows:
        return None
    return pd.DataFrame(rows, columns=['Name', 'Start', 'Duration'])


def _events_df_from_csv(csv_path):
    """Parse a Compumedics event CSV -> DataFrame Name/Start/Duration, or None."""
    df = pd.read_csv(str(csv_path))
    cols = {c.lower(): c for c in df.columns}
    if 'name' in cols and 'start' in cols and 'duration' in cols:
        return df.rename(columns={cols['name']: 'Name', cols['start']: 'Start',
                                  cols['duration']: 'Duration'})[['Name', 'Start', 'Duration']]
    return None


def _events_df_from_xml(xml_path):
    """Parse the <ScoredEvents> of a Profusion CMPStudyConfig .edf.XML
    -> DataFrame Name/Start/Duration (seconds) + Desaturation (depth in %, NaN when the
    event has no <Desaturation> child - only SpO2 desaturations carry one), or None.
    Tool 6's parser plus the additive Desaturation column (needed by the ODI threshold)."""
    root = ET.parse(str(xml_path)).getroot()
    rows = []
    for se in root.iter('ScoredEvent'):
        name_el = se.find('Name')
        if name_el is None or name_el.text is None:
            continue
        start_el = se.find('Start')
        dur_el = se.find('Duration')
        desat_el = se.find('Desaturation')
        start = float(start_el.text) if (start_el is not None and start_el.text) else np.nan
        dur = float(dur_el.text) if (dur_el is not None and dur_el.text) else np.nan
        try:
            desat = float(desat_el.text) if (desat_el is not None and desat_el.text) else np.nan
        except ValueError:
            desat = np.nan
        rows.append((name_el.text.strip(), start, dur, desat))
    if not rows:
        return None
    return pd.DataFrame(rows, columns=['Name', 'Start', 'Duration', 'Desaturation'])


def _events_df_from_manual(manual_path):
    """Read a tool-7 manual-annotation table ({stem}_manual_events.tsv: type / onset_s / duration_s,
    written by the reviewer in tool 7's navigator) -> the same Name/Start/Duration DataFrame as the
    scored companions, or None. These labels are ALREADY canonical (see augment_remap_for_manual)."""
    df = pd.read_csv(manual_path, sep='\t')
    if not {'type', 'onset_s'}.issubset(df.columns):
        return None
    out = pd.DataFrame({'Name': df['type'].astype(str),
                        'Start': pd.to_numeric(df['onset_s'], errors='coerce'),
                        'Duration': (pd.to_numeric(df['duration_s'], errors='coerce')
                                     if 'duration_s' in df.columns else np.nan),
                        'Source': 'manual'})   # tags the rows augment_remap_for_manual may map by identity
    out = out[out['Start'].notna()]
    return out if len(out) else None


def augment_remap_for_manual(event_remap, events_df):
    """Manual annotations already carry a CANONICAL label (tool 7 offers the canonical vocabulary), but
    compute_event_epoch_mask maps every name through event_remap.get(name) and would drop them as
    unknown. Return a copy of the remap with an identity entry per manual label. ONLY the rows tagged
    Source == 'manual' are treated this way: a SCORED label missing from the remap stays unmapped on
    purpose (the user chose not to map it in tool 4). The event_remap.json file itself is never
    rewritten — this lives in memory, for this run only."""
    if events_df is None or not len(events_df) or 'Source' not in events_df.columns:
        return event_remap
    out = dict(event_remap)
    manual = events_df[events_df['Source'].astype(str) == 'manual']
    for name in manual['Name'].astype(str).unique():
        out.setdefault(name, name)
    return out


def load_events(edf_path, txt_suffix='_ScoredEvents_Export.txt', csv_suffix='_event_xml.csv',
                manual_suffix='_manual_events.tsv', include_manual=False):
    """Load scored events next to the EDF, TXT-first then CSV then *.edf.XML fallback.
    Returns (DataFrame Name/Start/Duration in seconds, source) with source in
    {'txt', 'csv', 'xml'}, or (None, None) when no usable event companion is found.
    The .txt onsets need the recording-start datetime (header offsets 168/176), so a .txt present
    without a readable start datetime is skipped in favour of the CSV/XML companions.

    With include_manual=True the tool-7 manual annotations ({stem}_manual_events.tsv) are APPENDED to
    whichever scored source won the priority chain (source tag e.g. 'csv+manual'), or used alone when
    there is no scored companion at all ('manual'). They are an ADDITION to the scored corpus, never an
    alternative source, so the priority chain above is untouched. Default False -> byte-identical to the
    pre-annotation tool."""
    txt, csv, xml = event_companion_paths(edf_path, txt_suffix, csv_suffix)
    df, source = None, None
    if txt is not None:
        rec_start = read_edf_start_datetime(edf_path)
        if rec_start is not None:
            try:
                d = _events_df_from_txt(txt, rec_start)
                if d is not None:
                    df, source = d, 'txt'
            except Exception:
                pass  # fall through to the CSV
    if df is None and csv is not None:
        try:
            d = _events_df_from_csv(csv)
            if d is not None:
                df, source = d, 'csv'
        except Exception:
            pass  # fall through to the XML
    if df is None and xml is not None:
        try:
            d = _events_df_from_xml(xml)
            if d is not None:
                df, source = d, 'xml'
        except Exception:
            pass
    if include_manual:
        # Tool-7 annotations are ADDED to whatever the chain above found (never a replacement).
        manual_path = Path(edf_path).with_name(f'{Path(edf_path).stem}{manual_suffix}')
        if manual_path.exists():
            try:
                m = _events_df_from_manual(manual_path)
                if m is not None:
                    df = m if df is None else pd.concat([df, m], ignore_index=True)
                    source = 'manual' if source is None else f'{source}+manual'
            except Exception:
                pass  # a broken annotation file must never cost us the scored events
    if df is None:
        return None, None
    return df, source


def load_event_remap(path):
    """Lenient loader for event_remap.json (strict parse, then repair one trailing comma
    before a closing } or ]). Returns {raw_label: canonical_label_or_None}."""
    with open(path, encoding='utf-8') as f:
        text = f.read()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return json.loads(re.sub(r',(\s*[}\]])', r'\1', text))




# ---------------------------------------------------------------------------
# Event indices (AHI family, PLM, arousals) and ODI
# ---------------------------------------------------------------------------
def canonical_event_class(canonical):
    """Map a canonical label (tool 4 vocabulary) to the class counted here, or None.
    `hypopnea*` and `arousal*` are matched by prefix: users type variants such as
    `hypopnea_obstructive` / `arousal_respiratory` and they all belong to the same index."""
    if canonical is None:
        return None
    c = str(canonical).strip().lower()
    if c in ('apnea_obstructive', 'apnea_central', 'apnea_mixed', 'plm', 'spo2_desaturation'):
        return c
    if c.startswith('hypopnea'):
        return 'hypopnea'
    if c.startswith('arousal'):
        return 'arousal'
    return None


def select_events_in_window(events_df, cls, off_idx, on_idx, epoch_s, mode='tib'):
    """Keep the events whose onset falls in the counting window.
    'tib': onset in [lights-off, lights-on); 'sleep_epochs': onset in a sleep epoch.
    Returns (df, n_outside_window, n_bad_onset)."""
    if events_df is None or not len(events_df):
        return events_df, 0, 0
    start = pd.to_numeric(events_df['Start'], errors='coerce').to_numpy(dtype=float)
    bad = ~np.isfinite(start)
    ep_idx = np.floor(np.where(bad, -1, start) / epoch_s).astype(int)
    inside = (~bad) & (ep_idx >= off_idx) & (ep_idx < on_idx)
    if mode == 'sleep_epochs':
        sleep = np.asarray(cls['sleep_mask'], dtype=bool)
        ok_idx = (ep_idx >= 0) & (ep_idx < len(sleep))
        in_sleep = np.zeros_like(inside)
        in_sleep[ok_idx] = sleep[ep_idx[ok_idx]]
        inside = inside & in_sleep
    n_outside = int(((~inside) & (~bad)).sum())
    return events_df[inside].copy(), n_outside, int(bad.sum())


def compute_event_indices(events_df, remap, tst_min):
    """Counts and per-hour indices of the scored events (already windowed).
    Returns (metrics dict, checks list, n_unmapped). A TST of 0 / NaN gives NaN indices."""
    keys = [k for k, _, _, g, _ in METRICS if g in ('respiratory', 'plm', 'arousal')]
    metrics = {k: np.nan for k in keys}
    checks = []
    if events_df is None:
        return metrics, checks, 0
    counts = {'apnea_obstructive': 0, 'apnea_central': 0, 'apnea_mixed': 0, 'hypopnea': 0,
              'plm': 0, 'arousal': 0}
    unmapped = {}
    for name in events_df['Name'].astype(str):
        canonical = (remap or {}).get(name)
        if canonical is None:
            if remap is None or name not in remap:      # absent from the remap (never seen by tool 4)
                unmapped[name] = unmapped.get(name, 0) + 1
            continue                                    # explicitly ignored (null) or unmapped
        klass = canonical_event_class(canonical)
        if klass in counts:
            counts[klass] += 1
    metrics.update({'n_apnea_obstructive': counts['apnea_obstructive'],
                    'n_apnea_central': counts['apnea_central'], 'n_apnea_mixed': counts['apnea_mixed'],
                    'n_hypopnea': counts['hypopnea'], 'n_plm': counts['plm'],
                    'n_arousal': counts['arousal']})
    tst_h = tst_min / 60.0 if (tst_min is not None and np.isfinite(tst_min) and tst_min > 0) else np.nan
    metrics['oai'] = counts['apnea_obstructive'] / tst_h
    metrics['cai'] = counts['apnea_central'] / tst_h
    metrics['mai'] = counts['apnea_mixed'] / tst_h
    metrics['hi'] = counts['hypopnea'] / tst_h
    metrics['ahi'] = metrics['oai'] + metrics['cai'] + metrics['mai'] + metrics['hi']
    metrics['plm_index'] = counts['plm'] / tst_h
    metrics['arousal_index'] = counts['arousal'] / tst_h
    n_unmapped = int(sum(unmapped.values()))
    if unmapped:
        listing = ', '.join(f'{k} x{v}' for k, v in sorted(unmapped.items(), key=lambda kv: -kv[1]))
        checks.append({'check': 'unmapped_events', 'status': 'warning',
                       'detail': f'{n_unmapped} event(s) with a label absent from event_remap.json '
                                 f'(not counted; map them with tool 4): {listing}'})
    return metrics, checks, n_unmapped


def compute_odi(events_df, remap, tst_min, threshold_pct, xml_path, cls, off_idx, on_idx, epoch_s,
                window_mode):
    """Oxygen desaturation index from the scored desaturations.
    When the events carry a `Desaturation` depth (XML source, or the XML companion re-read for
    that purpose) only the events with depth >= threshold are counted; otherwise every scored
    desaturation is counted and the rule says so. Returns (metrics dict, checks, rule)."""
    metrics = {'n_desaturation': np.nan, 'odi': np.nan}
    checks = []
    df = events_df
    rule = 'all_events'
    has_depth = df is not None and 'Desaturation' in df.columns and df['Desaturation'].notna().any()
    if not has_depth and xml_path is not None:
        try:
            xdf = _events_df_from_xml(xml_path)
            if xdf is not None and xdf['Desaturation'].notna().any():
                xdf, _, _ = select_events_in_window(xdf, cls, off_idx, on_idx, epoch_s, window_mode)
                df, has_depth = xdf, True
                checks.append({'check': 'odi_rule', 'status': 'info',
                               'detail': 'desaturation depths read from the .edf.XML companion'})
        except Exception as exc:
            checks.append({'check': 'odi_rule', 'status': 'warning',
                           'detail': f'could not read the XML desaturation depths ({exc})'})
    if df is None:
        return metrics, checks, rule
    is_desat = np.array([canonical_event_class((remap or {}).get(str(n))) == 'spo2_desaturation'
                         for n in df['Name']], dtype=bool)
    n_all = int(is_desat.sum())
    if has_depth:
        depth = pd.to_numeric(df['Desaturation'], errors='coerce').to_numpy(dtype=float)
        deep = is_desat & np.isfinite(depth) & (depth >= threshold_pct)
        # depth unknown for a desaturation event -> counted (benefit of the doubt), reported
        no_depth = is_desat & ~np.isfinite(depth)
        n = int(deep.sum() + no_depth.sum())
        rule = f'depth>={threshold_pct:g}%'
        if no_depth.any():
            checks.append({'check': 'odi_rule', 'status': 'warning',
                           'detail': f'{int(no_depth.sum())} desaturation(s) without a depth value '
                                     f'were counted regardless of the threshold'})
    else:
        n = n_all
        if n_all:
            checks.append({'check': 'odi_rule', 'status': 'warning',
                           'detail': 'no desaturation depth available (txt/csv events, no XML) - '
                                     'every scored desaturation counted, threshold not applied'})
    metrics['n_desaturation'] = n
    tst_h = tst_min / 60.0 if (tst_min is not None and np.isfinite(tst_min) and tst_min > 0) else np.nan
    metrics['odi'] = n / tst_h
    return metrics, checks, rule


# ---------------------------------------------------------------------------
# SpO2 signal -> T90
# ---------------------------------------------------------------------------
def load_spo2_signal(edf_path, ch_name):
    """Read ONE channel (`include=` at read time) -> (values in %, sfreq). A channel stored as a
    fraction (max <= 1) is rescaled to %, which the caller reports."""
    raw = mne.io.read_raw_edf(str(edf_path), include=[ch_name], preload=True, verbose=False)
    x = raw.get_data()[0].astype(float)
    sf = float(raw.info['sfreq'])
    rescaled = False
    finite = x[np.isfinite(x)]
    if len(finite) and np.nanmax(finite) <= 1.0:
        x = x * 100.0
        rescaled = True
    return x, sf, rescaled


def compute_t90(x, sfreq, off_idx, on_idx, epoch_s, tst_min, thr=90.0, floor=50.0):
    """Time below `thr` % between lights-off and lights-on, on the valid samples only
    (a sample is artefact when <= 0 or below `floor` - sensor off, motion). Also the mean, the
    nadir and the artefact percentage. Returns (metrics dict, checks)."""
    metrics = {'t90_min': np.nan, 't90_pct_tst': np.nan, 'spo2_mean_pct': np.nan,
               'spo2_nadir_pct': np.nan, 'spo2_artifact_pct': np.nan}
    checks = []
    i0 = int(round(off_idx * epoch_s * sfreq))
    i1 = int(round(on_idx * epoch_s * sfreq))
    if i1 > len(x):
        checks.append({'check': 'spo2_length', 'status': 'warning',
                       'detail': f'SpO2 signal ends {(i1 - len(x)) / sfreq:.0f} s before lights-on - '
                                 f'computed on the available samples'})
        i1 = len(x)
    seg = x[i0:i1]
    if len(seg) == 0:
        checks.append({'check': 'spo2_length', 'status': 'fail', 'detail': 'no SpO2 sample in bed'})
        return metrics, checks
    valid = np.isfinite(seg) & (seg > 0) & (seg >= floor)
    n_valid = int(valid.sum())
    metrics['spo2_artifact_pct'] = 100.0 * (1 - n_valid / len(seg))
    if n_valid == 0:
        checks.append({'check': 'spo2_artifact', 'status': 'fail',
                       'detail': 'every SpO2 sample in bed is artefact (<= 0 or below the floor)'})
        return metrics, checks
    below = valid & (seg < thr)
    metrics['t90_min'] = float(below.sum()) / sfreq / 60.0
    metrics['t90_pct_tst'] = (100.0 * metrics['t90_min'] / tst_min
                              if (tst_min is not None and np.isfinite(tst_min) and tst_min > 0) else np.nan)
    metrics['spo2_mean_pct'] = float(np.mean(seg[valid]))
    metrics['spo2_nadir_pct'] = float(np.min(seg[valid]))
    if metrics['spo2_artifact_pct'] > 10:
        checks.append({'check': 'spo2_artifact', 'status': 'warning',
                       'detail': f"{metrics['spo2_artifact_pct']:.1f} % of the SpO2 samples in bed "
                                 f'discarded as artefact - T90 may be underestimated'})
    return metrics, checks


# ---------------------------------------------------------------------------
# Consistency checks and output tables
# ---------------------------------------------------------------------------
def run_checks(metrics, extras, lights, n_hyp, cls, event_info):
    """Arithmetic and plausibility checks on one participant's metrics.
    Returns a list of {'check', 'status', 'detail'} (status: ok / info / warning / fail)."""
    out = []

    def add(check, ok, detail, bad='fail'):
        out.append({'check': check, 'status': 'ok' if ok else bad, 'detail': detail})

    m = metrics
    has_sleep = np.isfinite(m.get('tst_min', np.nan))
    if has_sleep:
        total = m['tst_min'] + m['waso_min'] + m['other_min']
        add('spt_consistency', abs(m['spt_min'] - total) < 1e-6,
            f"SPT {m['spt_min']:.1f} = TST {m['tst_min']:.1f} + WASO {m['waso_min']:.1f}"
            + (f" + other {m['other_min']:.1f}" if m['other_min'] > 0 else ''))
        add('se_range', 0 <= m['se_pct'] <= 100 + 1e-9, f"SE = {m['se_pct']:.1f} %")
        add('tst_le_spt_le_tib', m['tst_min'] <= m['spt_min'] + 1e-9 <= m['tib_min'] + 1e-9,
            f"TST {m['tst_min']:.1f} <= SPT {m['spt_min']:.1f} <= TIB {m['tib_min']:.1f}")
        add('sol_plus_spt_le_tib', m['sol_min'] + m['spt_min'] <= m['tib_min'] + 1e-9,
            f"SOL {m['sol_min']:.1f} + SPT {m['spt_min']:.1f} <= TIB {m['tib_min']:.1f}")
        pct_sum = m['n1_pct'] + m['n2_pct'] + m['n3_pct'] + m['rem_pct']
        add('stage_pct_sum', abs(pct_sum - 100) < 0.1, f'N1+N2+N3+REM = {pct_sum:.2f} %')
        lats = {k: m[k] for k in ('lat_n1_min', 'lat_n2_min', 'lat_n3_min', 'lat_rem_min')}
        bad = [k for k, v in lats.items() if np.isfinite(v) and not (0 <= v <= m['spt_min'])]
        add('latencies_range', not bad,
            'every latency within [0, SPT]' if not bad else f'out of range: {", ".join(bad)}')
        missing = [k for k, v in lats.items() if not np.isfinite(v)]
        if missing:
            out.append({'check': 'stage_absent', 'status': 'info',
                        'detail': 'stage never reached (latency NaN): '
                                  + ', '.join(k.replace('lat_', '').replace('_min', '').upper()
                                              for k in missing)})
    add('lights_order', 0 <= lights['off_idx'] < lights['on_idx'] <= n_hyp,
        f"lights-off epoch {lights['off_idx']} < lights-on epoch {lights['on_idx']} <= {n_hyp}")
    n_other = int(cls['other_mask'].sum())
    if n_other or cls['unknown_labels']:
        labels = ', '.join(cls['unknown_labels']) if cls['unknown_labels'] else 'MT / custom'
        policy = ('counted in TIB and SPT, excluded from TST and WASO (YASA convention)'
                  if OTHER_STAGE_POLICY == 'exclude' else 'counted as wake (WASO inside SPT)')
        out.append({'check': 'mt_custom_present', 'status': 'warning',
                    'detail': f'{n_other} epoch(s) neither sleep nor wake ({labels}): {policy}. '
                              f'Re-map them with tool 3 if they should be scored as a stage.'})
    if event_info.get('source') is None and event_info.get('enabled'):
        out.append({'check': 'events_source', 'status': 'info',
                    'detail': 'no scored-event companion found - event indices are NaN'})
    if event_info.get('n_bad_onset'):
        out.append({'check': 'event_onsets_in_recording', 'status': 'warning',
                    'detail': f"{event_info['n_bad_onset']} event(s) with a missing / unreadable onset "
                              f'were skipped'})
    if event_info.get('n_outside'):
        out.append({'check': 'events_outside_window', 'status': 'info',
                    'detail': f"{event_info['n_outside']} event(s) outside the counting window "
                              f'({EVENT_COUNT_WINDOW}) were not counted'})
    return out


def metrics_long_df(file_id, metrics, provenance):
    """Long table: one row per registered metric (NaN when not computed) + provenance rows."""
    rows = []
    for key, label, unit, group, definition in METRICS:
        val = metrics.get(key, np.nan)
        rows.append({'file_id': file_id, 'group': group, 'metric': key,
                     'value': (round(float(val), 4) if val is not None and np.isfinite(float(val))
                               else np.nan),
                     'unit': unit, 'definition_short': definition})
    for key in PROVENANCE_KEYS:
        rows.append({'file_id': file_id, 'group': 'provenance', 'metric': key,
                     'value': provenance.get(key, ''), 'unit': '', 'definition_short': ''})
    return pd.DataFrame(rows, columns=['file_id', 'group', 'metric', 'value', 'unit', 'definition_short'])


def checks_df(file_id, checks):
    return pd.DataFrame([{'file_id': file_id, **c} for c in checks],
                        columns=['file_id', 'check', 'status', 'detail'])


def long_to_wide(long_df):
    """Pivot the concatenated long tables back to one row per participant: metric columns as
    float (METRICS order), provenance columns as text, in a fixed order."""
    if long_df is None or not len(long_df):
        return pd.DataFrame(columns=['file_id'] + METRIC_KEYS + PROVENANCE_KEYS)
    rows = []
    for fid, g in long_df.groupby('file_id', sort=True):
        vals = dict(zip(g['metric'], g['value']))
        row = {'file_id': fid}
        for k in METRIC_KEYS:
            row[k] = pd.to_numeric(pd.Series([vals.get(k)]), errors='coerce').iloc[0]
        for k in PROVENANCE_KEYS:
            v = vals.get(k, '')
            row[k] = '' if (v is None or (isinstance(v, float) and np.isnan(v))) else v
        rows.append(row)
    return pd.DataFrame(rows, columns=['file_id'] + METRIC_KEYS + PROVENANCE_KEYS)


# ---------------------------------------------------------------------------
# Comparison with an external reference table (Section 5)
# ---------------------------------------------------------------------------
# Aliases used to guess which reference column holds which metric (normalised: lower-case,
# accents stripped, non-alphanumerics removed). Extend freely.
METRIC_ALIASES = {
    'tib_min': ['tib', 'timeinbed', 'tempsaulit', 'tempsaulitmin', 'tpsaulit'],
    'sol_min': ['sol', 'sleeponsetlatency', 'sleeplatency', 'latenceendormissement', 'latencedendormissement', 'latencesommeil'],
    'spt_min': ['spt', 'sleepperiodtime', 'periodedesommeil', 'tempsperiodesommeil'],
    'tst_min': ['tst', 'totalsleeptime', 'tempstotaldesommeil', 'tempsdesommeiltotal', 'tts'],
    'waso_min': ['waso', 'wakeaftersleeponset', 'eveilintrasommeil', 'eveilapresendormissement'],
    'se_pct': ['se', 'sepct', 'sleepefficiency', 'sleepefficiencypct', 'efficacitedusommeil',
               'efficacitesommeil', 'efficacitesommeilpct', 'effsommeil', 'effsommeilpct', 'efficacite'],
    'sme_pct': ['sme', 'sleepmaintenanceefficiency'],
    'lat_n1_min': ['n1latency', 'latencyn1', 'latencen1'],
    'lat_n2_min': ['n2latency', 'latencyn2', 'latencen2'],
    'lat_n3_min': ['n3latency', 'latencyn3', 'latencen3', 'swslatency', 'latencesws'],
    'lat_rem_min': ['remlatency', 'latencyrem', 'latencerem', 'latencesp', 'latenceparadoxal'],
    'n1_min': ['n1min', 'n1duration', 'n1'],
    'n2_min': ['n2min', 'n2duration', 'n2'],
    'n3_min': ['n3min', 'n3duration', 'n3', 'sws', 'swsmin'],
    'rem_min': ['remmin', 'remduration', 'rem', 'sp', 'spmin'],
    'n1_pct': ['n1pct', 'n1percent', 'n1tst', 'pctn1', 'n1ptst'],
    'n2_pct': ['n2pct', 'n2percent', 'n2tst', 'pctn2', 'n2ptst'],
    'n3_pct': ['n3pct', 'n3percent', 'n3tst', 'pctn3', 'n3ptst', 'swspct'],
    'rem_pct': ['rempct', 'rempercent', 'remtst', 'pctrem', 'remptst', 'sppct'],
    'slow_sleep_min': ['slowsleep', 'slowsleepmin', 'n2n3', 'n2n3min'],
    'slow_sleep_pct': ['slowsleeppct', 'n2n3pct'],
    'ahi': ['ahi', 'iah', 'apneahypopneaindex', 'indexapneeshypopnees'],
    'oai': ['oai', 'obstructiveapneaindex', 'iao'],
    'cai': ['cai', 'centralapneaindex', 'iac'],
    'mai': ['mai', 'mixedapneaindex', 'iam'],
    'hi': ['hi', 'hypopneaindex', 'indexhypopnees', 'ih'],
    'n_apnea_obstructive': ['obstructiveapneas', 'apneesobstructives', 'nobstructiveapnea'],
    'n_apnea_central': ['centralapneas', 'apneescentrales', 'ncentralapnea'],
    'n_apnea_mixed': ['mixedapneas', 'apneesmixtes', 'nmixedapnea'],
    'n_hypopnea': ['hypopneas', 'hypopnees', 'nhypopnea'],
    'plm_index': ['plmindex', 'plmi', 'indexplm', 'mpjindex', 'indexmpj', 'plm'],
    'n_plm': ['nplm', 'plmcount', 'nombreplm'],
    'arousal_index': ['arousalindex', 'ai', 'indexmicroeveils', 'microeveilsindex', 'indexeveils'],
    'n_arousal': ['arousals', 'narousal', 'microeveils', 'nombremicroeveils'],
    'odi': ['odi', 'oxygendesaturationindex', 'indexdesaturation', 'ido', 'odi3', 'odi4'],
    'n_desaturation': ['desaturations', 'ndesaturation', 'nombredesaturations'],
    't90_min': ['t90', 't90min', 'tempsspo290', 'timebelow90'],
    't90_pct_tst': ['t90pct', 't90percent', 'ct90'],
    'spo2_mean_pct': ['meanspo2', 'spo2mean', 'spo2moyenne', 'saturationmoyenne'],
    'spo2_nadir_pct': ['nadirspo2', 'spo2nadir', 'minspo2', 'spo2min', 'saturationminimale'],
}


def _norm_name(text):
    """'Eff. sommeil %' -> 'effsommeilpct' (a % sign is kept as 'pct' so 'N3 %' cannot match 'N3')."""
    return re.sub(r'[^a-z0-9]', '', strip_accents(str(text).replace('%', ' pct ')).lower())


def guess_metric_mapping(ref_columns):
    """{metric key: reference column or '(none)'}: exact alias match first, then a fuzzy match on
    the normalised names (cutoff 0.8 so 'n1' never steals 'n1pct'); each column used once."""
    cols = [str(c) for c in ref_columns]
    normed = {_norm_name(c): c for c in cols}
    mapping, used = {}, set()
    for key in METRIC_KEYS:                                # exact alias pass
        aliases = [_norm_name(key)] + METRIC_ALIASES.get(key, [])
        hit = next((normed[a] for a in aliases if a in normed and normed[a] not in used), None)
        if hit is not None:
            mapping[key], used = hit, used | {hit}
    for key in METRIC_KEYS:                                # fuzzy pass for what is left
        if key in mapping:
            continue
        aliases = [_norm_name(key)] + METRIC_ALIASES.get(key, [])
        best = None
        for a in aliases:
            cands = difflib.get_close_matches(a, [n for n, c in normed.items() if c not in used],
                                              n=1, cutoff=0.8)
            if cands:
                best = normed[cands[0]]
                break
        mapping[key] = best if best is not None else '(none)'
        if best is not None:
            used.add(best)
    return mapping


def default_tolerance(key):
    """Absolute tolerance by unit: 1 min, 1 %-point, 1 event/h, exact for counts."""
    unit = METRIC_INFO[key]['unit']
    return 0.0 if unit == 'count' else 1.0


def compare_with_reference(wide_df, ref_df, id_col, mapping, tolerances):
    """One row per (participant, mapped metric): our value, the reference value, the difference
    and a status - match / mismatch (|diff| > tol) / missing_ours / missing_reference /
    not_in_reference_table (participant absent from the reference)."""
    ours = wide_df.copy()
    ours['_key'] = ours['file_id'].astype(str).map(norm)
    ref = ref_df.copy()
    ref['_key'] = ref[id_col].astype(str).str.strip().map(norm)
    ref = ref.drop_duplicates('_key')
    ref_keys = set(ref['_key'])
    ref_idx = ref.set_index('_key')
    rows = []
    for _, r in ours.iterrows():
        in_ref = r['_key'] in ref_keys
        for key, col in mapping.items():
            if col in (None, '(none)'):
                continue
            our_v = pd.to_numeric(pd.Series([r.get(key)]), errors='coerce').iloc[0]
            ref_v = (pd.to_numeric(pd.Series([ref_idx.loc[r['_key'], col]]), errors='coerce').iloc[0]
                     if in_ref else np.nan)
            tol = float(tolerances.get(key, default_tolerance(key)))
            if not in_ref:
                status = 'not_in_reference_table'
            elif not np.isfinite(our_v) and not np.isfinite(ref_v):
                status = 'missing_both'
            elif not np.isfinite(our_v):
                status = 'missing_ours'
            elif not np.isfinite(ref_v):
                status = 'missing_reference'
            else:
                status = 'match' if abs(our_v - ref_v) <= tol + 1e-9 else 'mismatch'
            rows.append({'file_id': r['file_id'], 'metric': key, 'ours': our_v, 'reference': ref_v,
                         'ref_column': col, 'diff': (our_v - ref_v) if np.isfinite(our_v) and np.isfinite(ref_v) else np.nan,
                         'tol': tol, 'status': status})
    return pd.DataFrame(rows, columns=['file_id', 'metric', 'ours', 'reference', 'ref_column',
                                       'diff', 'tol', 'status'])

In [ ]:
# =============================================================================
# Figures and HTML blocks of the database report
# =============================================================================
STATUS_COLORS = {'ok': '#2e7d32', 'info': '#1565c0', 'warning': '#ef6c00', 'fail': '#c62828'}
SOURCE_COLORS = {'summary_txt': '#1b9e77', 'light_channel': '#7570b3', 'participant_table': '#d95f02',
                 'recording_bounds': '#999999'}


def params_html(cfg):
    """Two-column 'parameter -> value' table."""
    rows = ''.join(f'<tr><td style="padding:2px 12px 2px 0"><b>{k}</b></td>'
                   f'<td>{v}</td></tr>' for k, v in cfg.items())
    return f'<table style="font-size:90%">{rows}</table>'


def save_report_html(path, title, items):
    """Write an mne.Report from an ORDERED list of ('fig'|'html', title, payload) items
    (`None` payloads are skipped)."""
    report = mne.Report(title=title, verbose=False)
    for kind, item_title, payload in items:
        if payload is None:
            continue
        if kind == 'fig':
            report.add_figure(fig=payload, title=item_title, image_format='PNG')
        else:
            report.add_html(html=payload, title=item_title)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    report.save(str(path), overwrite=True, open_browser=False, verbose=False)
    return path


def close_figs(items):
    for kind, _, payload in items:
        if kind == 'fig' and payload is not None:
            plt.close(payload)


def _shade_reference(ax, key, horizontal=False):
    """Shade the indicative normal range of a metric (and the AHI severity bands)."""
    if key == 'ahi' or key == 'odi':
        colors = ['#e8f5e9', '#fff8e1', '#ffe0b2', '#ffcdd2']
        if not horizontal:
            ax.set_ylim(min(ax.get_ylim()[0], 0), max(ax.get_ylim()[1], 6))
        top = ax.get_ylim()[1] if not horizontal else ax.get_xlim()[1]
        for (lo, hi, name), col in zip(AHI_SEVERITY_BANDS, colors):
            hi_v = hi if hi is not None else max(top, lo + 5)
            if horizontal:
                ax.axvspan(lo, hi_v, color=col, zorder=0)
                ax.text((lo + hi_v) / 2, ax.get_ylim()[1], name, ha='center', va='top', fontsize=7, color='#555')
            else:
                ax.axhspan(lo, hi_v, color=col, zorder=0)
                ax.text(ax.get_xlim()[1], (lo + hi_v) / 2, name, ha='right', va='center', fontsize=7, color='#555')
        return
    rng = REFERENCE_RANGES.get(key)
    if rng is None:
        return
    lo, hi, _ = rng
    if horizontal:
        ax.axvspan(lo, hi, color='#e8f5e9', zorder=0)
    else:
        ax.axhspan(lo, hi, color='#e8f5e9', zorder=0)
        y0, y1 = ax.get_ylim()                 # keep the band visible even when every point is outside
        ax.set_ylim(min(y0, lo), max(y1, hi))


def plot_metric_distributions(wide, groups, title):
    """One panel per metric of `groups` with at least one value: every participant as a dot
    (jittered) over a boxplot, the indicative normal range shaded green. None when empty."""
    keys = [k for k in METRIC_KEYS if METRIC_INFO[k]['group'] in groups
            and k in wide.columns and pd.to_numeric(wide[k], errors='coerce').notna().any()]
    if not keys:
        return None
    n_cols = min(4, len(keys))
    n_rows = int(np.ceil(len(keys) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.4 * n_cols, 3.0 * n_rows), squeeze=False)
    rng = np.random.default_rng(0)
    for ax, key in zip(axes.ravel(), keys):
        vals = pd.to_numeric(wide[key], errors='coerce').dropna().to_numpy(dtype=float)
        ax.boxplot(vals, positions=[0], widths=0.5, showfliers=False,
                   medianprops={'color': '#c62828'}, zorder=2)
        ax.scatter(rng.uniform(-0.15, 0.15, len(vals)), vals, s=14, color='#455a64', alpha=0.75, zorder=3)
        ax.set_xlim(-0.6, 0.6)
        ax.set_xticks([])
        ax.set_title(METRIC_INFO[key]['label'], fontsize=9)
        ax.set_ylabel(METRIC_INFO[key]['unit'], fontsize=8)
        ax.tick_params(labelsize=8)
        _shade_reference(ax, key)
        ax.text(0.02, 0.98, f'n = {len(vals)}', transform=ax.transAxes, ha='left', va='top',
                fontsize=7, color='#555')
    for ax in axes.ravel()[len(keys):]:
        ax.axis('off')
    fig.suptitle(f'{title} - green band = indicative normal range (healthy adults)', fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    return fig


def _clock_to_hours(text):
    """'23:15:00' -> 23.25; after-midnight times shifted by +24 so the night reads left to right."""
    try:
        hh, mm, ss = (int(x) for x in str(text).split(':'))
    except Exception:
        return np.nan
    h = hh + mm / 60 + ss / 3600
    return h + 24 if h < 12 else h


def plot_lights_overview(wide):
    """Per participant: the in-bed span (lights-off -> lights-on, coloured by the lights source)
    and the sleep period (sleep onset -> last sleep epoch, dark). Clock time on the x axis."""
    if wide is None or not len(wide) or 'lights_off_clock' not in wide.columns:
        return None
    df = wide.copy()
    df['off_h'] = df['lights_off_clock'].map(_clock_to_hours)
    df['on_h'] = df['lights_on_clock'].map(_clock_to_hours)
    df = df[df['off_h'].notna() & df['on_h'].notna()]
    if not len(df):
        return None
    df = df.sort_values('off_h').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(11, max(2.5, 0.28 * len(df) + 1.2)))
    for i, r in df.iterrows():
        src = r.get('lights_source', 'recording_bounds')
        ax.barh(i, r['on_h'] - r['off_h'], left=r['off_h'], height=0.6,
                color=SOURCE_COLORS.get(src, '#999999'), alpha=0.45, edgecolor='none')
        sol = pd.to_numeric(pd.Series([r.get('sol_min')]), errors='coerce').iloc[0]
        spt = pd.to_numeric(pd.Series([r.get('spt_min')]), errors='coerce').iloc[0]
        if np.isfinite(sol) and np.isfinite(spt):
            ax.barh(i, spt / 60, left=r['off_h'] + sol / 60, height=0.35, color='#37474f')
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df['file_id'].astype(str), fontsize=7)
    ax.invert_yaxis()
    lo = np.floor(df['off_h'].min())
    hi = np.ceil(df['on_h'].max())
    ticks = np.arange(lo, hi + 1)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f'{int(t) % 24:02d}:00' for t in ticks], fontsize=8)
    ax.set_xlabel('clock time')
    ax.grid(axis='x', alpha=0.3)
    handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.45) for c in SOURCE_COLORS.values()]
    handles.append(plt.Rectangle((0, 0), 1, 1, color='#37474f'))
    ax.legend(handles, list(SOURCE_COLORS.keys()) + ['sleep period (onset -> last sleep epoch)'],
              fontsize=7, loc='upper left', bbox_to_anchor=(1.01, 1.0), title='lights source',
              title_fontsize=7)
    ax.set_title('Time in bed and sleep period per participant', fontsize=10)
    fig.tight_layout()
    return fig


def plot_comparison(cmp_df):
    """Ours vs reference, one panel per compared metric, identity line, mismatches in red."""
    if cmp_df is None or not len(cmp_df):
        return None
    keys = [k for k in METRIC_KEYS if k in set(cmp_df['metric'])]
    keys = [k for k in keys if cmp_df.loc[cmp_df['metric'] == k, 'status'].isin(['match', 'mismatch']).any()]
    if not keys:
        return None
    n_cols = min(4, len(keys))
    n_rows = int(np.ceil(len(keys) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.4 * n_cols, 3.2 * n_rows), squeeze=False)
    for ax, key in zip(axes.ravel(), keys):
        d = cmp_df[(cmp_df['metric'] == key) & cmp_df['status'].isin(['match', 'mismatch'])]
        col = d['status'].map({'match': '#2e7d32', 'mismatch': '#c62828'})
        ax.scatter(d['reference'], d['ours'], s=16, c=col, alpha=0.8, zorder=3)
        both = np.concatenate([d['reference'].to_numpy(float), d['ours'].to_numpy(float)])
        both = both[np.isfinite(both)]
        if len(both):
            lo, hi = both.min(), both.max()
            pad = 0.05 * (hi - lo if hi > lo else 1)
            ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color='#999', lw=0.8, zorder=1)
        n_mis = int((d['status'] == 'mismatch').sum())
        ax.set_title(f"{METRIC_INFO[key]['label']}\n{len(d) - n_mis} match / {n_mis} mismatch", fontsize=8)
        ax.set_xlabel('reference', fontsize=8)
        ax.set_ylabel('this tool', fontsize=8)
        ax.tick_params(labelsize=7)
    for ax in axes.ravel()[len(keys):]:
        ax.axis('off')
    fig.suptitle('Comparison with the reference table (red = beyond tolerance)', fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


def checks_summary_html(checks):
    """Count of participants per (check, status) + the list of participants with a failure."""
    if checks is None or not len(checks):
        return '<p><i>(no checks)</i></p>'
    piv = (checks.groupby(['check', 'status'])['file_id'].nunique().unstack('status')
           .reindex(columns=[s for s in ('ok', 'info', 'warning', 'fail')]).fillna(0).astype(int))
    piv = piv.reset_index()
    html = piv.to_html(index=False, border=0)
    fails = checks[checks['status'] == 'fail']
    if len(fails):
        items = ''.join(f'<li><b>{r.file_id}</b> - {r.check}: {r.detail}</li>' for r in fails.itertuples())
        html += f'<p style="color:#c62828"><b>Failures:</b></p><ul>{items}</ul>'
    warns = checks[checks['status'] == 'warning']
    if len(warns):
        counts = warns.groupby('check')['file_id'].nunique().sort_values(ascending=False)
        html += ('<p><b>Warnings</b> (participants affected): '
                 + ', '.join(f'{c} ({n})' for c, n in counts.items()) + '</p>')
    return html


def participants_html(wide):
    """Compact per-participant table of the headline metrics + provenance."""
    if wide is None or not len(wide):
        return '<p><i>(no participant)</i></p>'
    cols = ['file_id', 'tib_min', 'sol_min', 'tst_min', 'waso_min', 'se_pct', 'lat_rem_min',
            'n1_pct', 'n2_pct', 'n3_pct', 'rem_pct', 'ahi', 'plm_index', 'arousal_index', 'odi',
            't90_pct_tst', 'lights_source', 'n_mt_custom', 'n_unmapped_events', 'n_warnings']
    cols = [c for c in cols if c in wide.columns]
    df = wide[cols].copy()
    for c in cols:
        if c in METRIC_INFO:
            df[c] = pd.to_numeric(df[c], errors='coerce').round(1)
    return df.to_html(index=False, border=0, na_rep='')


def glossary_html():
    rows = ''
    for key, label, unit, group, definition in METRICS:
        rng = REFERENCE_RANGES.get(key)
        rtxt = f'{rng[0]}-{rng[1]} {unit}' + (f' <small>({rng[2]})</small>' if rng[2] else '') if rng else ''
        rows += (f'<tr><td><b>{label}</b><br><small><code>{key}</code></small></td><td>{unit}</td>'
                 f'<td>{definition}</td><td>{rtxt}</td></tr>')
    return ('<p>Indicative ranges are textbook values for healthy adults recorded in a laboratory; '
            'they depend on age, medication, the first-night effect and the scoring rules. Use them '
            'as a sanity check, never as a diagnostic cut-off.</p>'
            '<table style="font-size:85%"><tr><th>Metric</th><th>Unit</th><th>Definition</th>'
            f'<th>Indicative range</th></tr>{rows}</table>')


def guidance_html(cfg, data_dir, reports_dir, has_subject_info=False, has_comparison=False):
    """'Where to look': which file answers which question."""
    join_note = (' The participant-info columns are joined on the right of every row, so the '
                 'table can be regrouped by any population variable (e.g. <code>groupby("group")'
                 '["se_pct"].describe()</code>).' if has_subject_info else
                 ' Select a participant table in Section 1 to have its columns joined on the right '
                 'of every row (grouping by population variables).')
    cmp_note = ('<li><b><code>comparison_with_reference.tsv</code></b> - one row per (participant, '
                'compared metric): <code>ours</code>, <code>reference</code>, <code>diff</code>, the '
                'tolerance and a <code>status</code> (<code>match</code> / <code>mismatch</code> / '
                '<code>missing_ours</code> / <code>missing_reference</code> / '
                '<code>not_in_reference_table</code>). Filter on <code>mismatch</code> first: a '
                'systematic offset on one metric usually means a definition difference (lights, '
                'sleep-onset rule, MT handling), a single outlier a scoring or file problem.</li>'
                if has_comparison else '')
    return f'''
<p><b>Start with</b> <code>{reports_dir / 'global_sleep_metrics.tsv'}</code> - one row per participant, the
table for statistics.</p>
<ul>
<li><b><code>global_sleep_metrics.tsv</code></b> - <code>file_id</code>, then one column per metric
(<code>{', '.join(METRIC_KEYS[:8])}, ...</code>; units in the glossary: minutes, %, events/h, counts),
then the provenance columns (<code>lights_source</code>, <code>lights_off_clock</code>,
<code>lights_on_clock</code>, <code>sleep_onset_rule</code>, <code>n_mt_custom</code>,
<code>event_source</code>, <code>n_unmapped_events</code>, <code>odi_rule</code>, <code>n_warnings</code>
...).{join_note} Before trusting a value: check <code>lights_source</code> (a
<code>recording_bounds</code> row has TIB = whole recording, so SOL and SE are biased), <code>n_warnings</code>
and <code>n_unmapped_events</code> (events absent from <code>event_remap.json</code> are NOT counted in
the AHI / PLM / arousal indices - map them with tool 4 and re-run with Skip unticked).</li>
<li><b><code>global_sleep_metrics_checks.tsv</code></b> - every consistency check of every participant
(<code>check</code>, <code>status</code> ok / info / warning / fail, <code>detail</code>). A
<code>fail</code> means a metric could not be trusted or computed (no sleep, hypnogram/EDF mismatch);
a <code>warning</code> flags a fallback or an unusual situation (lights fallback, MT epochs, unmapped
events, SpO2 artefact ...).</li>
<li><b><code>sleep_metrics_database.xlsx</code></b> - the same tables as sheets
(<code>sleep_metrics</code>, <code>checks</code>, <code>glossary</code>, <code>parameters</code>,
<code>failed</code>); the TSV files stay the source of truth.</li>
<li><b><code>sleep_metrics_failed.tsv</code></b> - participants that produced no metrics and why
(missing hypnogram, length mismatch, unreadable EDF header).</li>
{cmp_note}
<li><b><code>{data_dir}/&lt;subtree&gt;/{{file_id}}_sleep_metrics.tsv</code></b> - the per-participant
long table (<code>metric</code>, <code>value</code>, <code>unit</code>, <code>definition_short</code>)
the global table is rebuilt from; <code>{{file_id}}_sleep_metrics_checks.tsv</code> next to it. Delete
a participant's <code>_sleep_metrics.tsv</code> to have it reprocessed on the next run.</li>
</ul>
<p><b>Parameters of this run</b>: lights source <code>{cfg.get('lights_source')}</code>, sleep-onset rule
<code>{cfg.get('onset_rule')}</code>, epoch length {cfg.get('epoch_s')} s, MT / custom policy
<code>{OTHER_STAGE_POLICY}</code>, event window <code>{EVENT_COUNT_WINDOW}</code>, ODI
{'on (threshold ' + str(cfg.get('odi_thr')) + ' %)' if cfg.get('odi') else 'off'}, T90
{'on (< ' + str(cfg.get('spo2_thr')) + ' %, floor ' + str(cfg.get('spo2_floor')) + ' %)' if cfg.get('t90') else 'off'}.</p>
'''

## 1 — Folders and scan

Point the tool at the **data folder** holding the EDF files (scanned recursively) with, next to each EDF,
the remapped hypnogram from tool 3 and — when available — the scored-event companions
(`*_ScoredEvents_Export.txt`, `*_event_xml.csv`, `*.edf.XML`) and the Profusion
`*_Summary_Export.txt` (lights-off/on). The suffixes are auto-detected and editable.

`event_remap.json` (tool 4) is picked up from `<data>/config_param/` when present; without it the
event indices (AHI, PLM, arousals, ODI) are NaN. The **output folder** is optional (default = the data
folder): the tool writes `derivatives/features_macrostructure/` and `reports_features_macrostructure/`
under it. The **participant table** is optional: its rows are joined to the global table on the column
you choose, and two of its columns can supply lights-off / lights-on (see Section 2).

**Scan** lists the participants, what was found next to each EDF, and how many are already processed.

In [ ]:
# =============================================================================
# Section 1 - folders, suffix auto-detection, scan
# =============================================================================
S = {}          # shared state across sections
_LBL = {'description_width': 'initial'}

fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = '<b>Data folder</b> (EDF files + hypnograms + event companions, scanned recursively):'

fc_out = FileChooser(os.getcwd())
fc_out.show_only_dirs = True
fc_out.title = ('<b>Output folder</b> &mdash; optional, default = the data folder (writes '
                '<code>derivatives/features_macrostructure/</code> and '
                '<code>reports_features_macrostructure/</code> under it):')

fc_remap = FileChooser(os.getcwd())
fc_remap.filter_pattern = ['*.json']
fc_remap.title = ('<b>Event remap</b> (<code>config_param/event_remap.json</code> from tool 4) &mdash; '
                  'auto-selected when present under the data folder:')

fc_subj = FileChooser(os.getcwd())
fc_subj.filter_pattern = ['*.csv', '*.tsv', '*.xlsx']
fc_subj.title = ('<b>Participant info</b> (optional <code>.csv</code>/<code>.tsv</code>/<code>.xlsx</code> '
                 'joined to the global table; may hold the lights-off/on columns):')
dd_join_col = widgets.Dropdown(description='Join on:', options=[], value=None,
                               layout=widgets.Layout(width='320px', display='none'), style=_LBL)

txt_hypno_suffix = widgets.Text(description='Hypnogram suffix:', value='_Hypnogram_remapped.txt',
                                layout=widgets.Layout(width='420px'), style=_LBL)
lbl_hypno_suffix = widgets.HTML('')
txt_summary_suffix = widgets.Text(description='Lights (summary) txt suffix:', value='_Summary_Export.txt',
                                  layout=widgets.Layout(width='420px'), style=_LBL)
lbl_summary_suffix = widgets.HTML('')
txt_evt_txt_suffix = widgets.Text(description='Event TXT suffix:', value='_ScoredEvents_Export.txt',
                                  layout=widgets.Layout(width='420px'), style=_LBL)
lbl_evt_txt_suffix = widgets.HTML('')
txt_evt_csv_suffix = widgets.Text(description='Event CSV suffix:', value='_event_xml.csv',
                                  layout=widgets.Layout(width='420px'), style=_LBL)
lbl_evt_csv_suffix = widgets.HTML('')
txt_custom = widgets.Text(description='Custom stages:', value='',
                          placeholder='comma-separated non-AASM labels kept by tool 3, e.g. N4',
                          layout=widgets.Layout(width='520px'), style=_LBL)

btn_scan = widgets.Button(description='Scan', button_style='primary', icon='search')
lbl_scan = widgets.HTML('<i>Pick the data folder, then Scan.</i>')
out_scan = widgets.Output()


def _list_edf(folder):
    return [f for f in sorted(Path(folder).rglob('*'))
            if f.suffix.lower() == '.edf' and not f.name.startswith('._')]


def _suffix_counts(edf_files, files):
    """{suffix: number of EDFs having a companion <stem><suffix>} over `files`."""
    counts = {}
    for edf in edf_files:
        for f in files:
            if norm(f.name).startswith(norm(edf.stem)):
                suf = f.name[len(edf.stem):]
                counts[suf] = counts.get(suf, 0) + 1
    return counts


def _detect_suffixes(edf_folder):
    """Auto-fill the four suffix fields from the files found next to the EDFs (tool 6's rules:
    hypnogram = longest suffix among those present for >= 50 % of the max count, skipping the
    event / summary / study-log exports; the exports = most frequent suffix, shortest on ties)."""
    edf_files = _list_edf(edf_folder)
    n_total = len(edf_files)
    if not edf_files:
        lbl_hypno_suffix.value = '<small style="color:#888">No EDF file found (recursive scan).</small>'
        return
    all_txt = [f for f in Path(edf_folder).rglob('*') if f.suffix.lower() == '.txt']
    all_csv = [f for f in Path(edf_folder).rglob('*') if f.suffix.lower() == '.csv']

    def fill(widget, label, counts, what, pick):
        if not counts:
            label.value = f'<small style="color:#888">No {what} detected next to the EDF files.</small>'
            return
        best, n = pick(counts)
        widget.value = best
        parts = [f'<b>{s}</b>&nbsp;(&times;{c}){"&nbsp;&larr; selected" if s == best else ""}'
                 for s, c in sorted(counts.items(), key=lambda x: -x[1])]
        color = '#2e7d32' if n == n_total else '#e67e00'
        label.value = (f'<small style="color:{color}">{what} detected: {"&nbsp;&middot;&nbsp;".join(parts)}'
                       f' &mdash; {n}/{n_total} files matching</small>')

    most_frequent = lambda c: min(c.items(), key=lambda x: (-x[1], len(x[0])))
    # hypnogram: exclude the exports from the SELECTION only, every suffix stays listed
    hyp_counts = _suffix_counts(edf_files, all_txt)
    sel = {s: c for s, c in hyp_counts.items()
           if not any(tok in s.lower() for tok in ('event', 'summary', 'studylog'))} or hyp_counts

    def pick_hypno(_):
        mx = max(sel.values())
        cands = {s: c for s, c in sel.items() if c >= mx * 0.5}
        return max(cands.items(), key=lambda x: (len(x[0]), x[1]))
    fill(txt_hypno_suffix, lbl_hypno_suffix, hyp_counts, 'Hypnogram .txt', pick_hypno)
    fill(txt_summary_suffix, lbl_summary_suffix,
         _suffix_counts(edf_files, [f for f in all_txt if 'summary' in f.name.lower()]),
         'Lights (summary) txt', most_frequent)
    fill(txt_evt_txt_suffix, lbl_evt_txt_suffix,
         _suffix_counts(edf_files, [f for f in all_txt if 'event' in f.name.lower()]),
         'Event TXT export', most_frequent)
    fill(txt_evt_csv_suffix, lbl_evt_csv_suffix, _suffix_counts(edf_files, all_csv),
         'Event CSV', most_frequent)


def _on_data(chooser):
    """Point the other choosers into the data folder, auto-select event_remap.json, detect suffixes."""
    try:
        data = fc_data.selected_path
        if not data:
            return
        data = Path(data)
        fc_out.reset(path=str(data))
        fc_subj.reset(path=str(data))
        remap_default = data / 'config_param' / 'event_remap.json'
        # the chooser is only pre-pointed; load_remap_file() falls back to this default
        # when the user does not click Select
        if remap_default.exists():
            fc_remap.reset(path=str(remap_default.parent), filename=remap_default.name)
        else:
            fc_remap.reset(path=str(data))
        cs = load_custom_stages(data)
        txt_custom.value = ', '.join(cs)
        _detect_suffixes(data)
        lbl_scan.value = '<i>Data folder set &mdash; check the suffixes, then Scan.</i>'
    except Exception as exc:
        lbl_scan.value = f'<span style="color:#c62828">Data folder error: {exc}</span>'


def _on_subj(chooser):
    """Load the optional participant table and offer its columns as the join key."""
    try:
        df, err = load_subject_info(fc_subj.selected)
        if df is None:
            S.pop('subj_info', None)
            dd_join_col.layout.display = 'none'
            lbl_scan.value = f'<span style="color:#ef6c00">Participant info unreadable: {err}</span>'
            return
        S['subj_info'] = df
        dd_join_col.options = list(df.columns)
        guess = [c for c in df.columns if norm(c) in ('file_id', 'fileid', 'sub_id', 'subid', 'id',
                                                       'participant', 'participant_id', 'subject')]
        dd_join_col.value = guess[0] if guess else list(df.columns)[0]
        dd_join_col.layout.display = ''
        lbl_scan.value = (f'<i>Participant info loaded ({len(df)} rows, columns: '
                          f'{", ".join(map(str, df.columns))}) &mdash; check the join column, then Scan.</i>')
    except Exception as exc:
        lbl_scan.value = f'<span style="color:#c62828">Participant info error: {exc}</span>'


def subject_row(file_id):
    """The participant-table row of a file_id (dict) or None."""
    info = S.get('subj_info')
    col = dd_join_col.value
    if info is None or not col or col not in info.columns:
        return None
    hit = info[info[col].astype(str).str.strip().map(norm) == norm(file_id)]
    return hit.iloc[0].to_dict() if len(hit) else None


def load_remap_file():
    """event_remap.json from the chooser, else the default under the data folder. (dict|None, note)."""
    path = fc_remap.selected
    if not path and fc_data.selected_path:
        cand = Path(fc_data.selected_path) / 'config_param' / 'event_remap.json'
        path = str(cand) if cand.exists() else None
    if not path or not Path(path).exists():
        return None, 'no event_remap.json - event labels cannot be counted (run tool 4)'
    try:
        return load_event_remap(path), f'event remap: {path} ({len(load_event_remap(path))} labels)'
    except Exception as exc:
        return None, f'event_remap.json unreadable ({exc})'


def scan_folder(_=None):
    """List the EDFs, read each header (no signal), locate the companions, count the epochs of
    each hypnogram, and flag what is already processed. Everything here is non-fatal."""
    with out_scan:
        clear_output()
        try:
            if not fc_data.selected_path:
                lbl_scan.value = '<span style="color:#c62828">Select the data folder first.</span>'
                return
            data_root = Path(fc_data.selected_path)
            out_root = Path(fc_out.selected_path) if fc_out.selected_path else data_root
            data_dir = out_root / 'derivatives' / DATA_DIRNAME
            reports_dir = out_root / REPORTS_DIRNAME
            edf_files = _list_edf(data_root)
            if not edf_files:
                lbl_scan.value = '<span style="color:#c62828">No EDF file found under that folder.</span>'
                return
            custom = parse_custom_field(txt_custom.value) or load_custom_stages(data_root)
            txt_custom.value = ', '.join(custom)
            remap, remap_note = load_remap_file()
            hypno_lookup = {norm(p.name): p for p in data_root.rglob('*') if p.suffix.lower() == '.txt'}
            hsuf, ssuf = txt_hypno_suffix.value, txt_summary_suffix.value
            tsuf, csuf = txt_evt_txt_suffix.value, txt_evt_csv_suffix.value

            parts, rows, warn = [], [], []
            n_done, n_light, n_spo2 = 0, 0, 0
            for edf in edf_files:
                fid = edf.stem
                p = {'file_id': fid, 'edf': edf, 'subtree': edf.parent.relative_to(data_root)}
                try:
                    header = read_edf_header_info(edf)
                except Exception as exc:
                    header = None
                    warn.append(f'{fid}: EDF header unreadable ({exc}) &mdash; will fail at run time')
                p['header'] = header
                hyp = hypno_lookup.get(norm(f'{fid}{hsuf}'))
                if hyp is None and (edf.with_name(f'{fid}{hsuf}')).exists():
                    hyp = edf.with_name(f'{fid}{hsuf}')
                p['hypno'] = hyp
                n_hyp = None
                if hyp is not None:
                    try:
                        n_hyp = len(load_hypnogram(hyp))
                    except Exception as exc:
                        warn.append(f'{fid}: hypnogram unreadable ({exc})')
                else:
                    warn.append(f'{fid}: no hypnogram <code>{fid}{hsuf}</code> &mdash; will be skipped')
                p['n_hypno'] = n_hyp
                summ = hypno_lookup.get(norm(f'{fid}{ssuf}'))
                p['summary_txt'] = summ
                txt, csv, xml = event_companion_paths(edf, tsuf, csuf)
                p.update({'evt_txt': txt, 'evt_csv': csv, 'evt_xml': xml})
                chs = header['ch_names'] if header else []
                p['light_ch'] = find_channel(chs, LIGHT_CH_PATTERN)
                p['spo2_ch'] = find_channel(chs, SPO2_CH_PATTERN)
                n_light += p['light_ch'] is not None
                n_spo2 += p['spo2_ch'] is not None
                p['out_data'] = data_dir / p['subtree']
                p['out_reports'] = reports_dir / p['subtree']
                done = (p['out_data'] / f'{fid}_sleep_metrics.tsv').exists()
                half = (not done) and (p['out_data'] / f'{fid}_sleep_metrics_checks.tsv').exists()
                p['done'] = done
                n_done += done
                if half:
                    warn.append(f'{fid}: checks file without metrics file (interrupted run) &mdash; '
                                f'will be reprocessed')
                n_edf_ep = int(np.floor(header['duration_s'] / DEFAULT_EPOCH_S)) if header else None
                if header and n_hyp is not None and abs(n_hyp - n_edf_ep) > 1:
                    warn.append(f'{fid}: hypnogram {n_hyp} epochs vs EDF {n_edf_ep} &mdash; mismatch > 1 '
                                f'epoch, will fail (wrong hypnogram or epoch length?)')
                lights_avail = ('txt' if summ is not None else
                                f'channel {p["light_ch"]}' if p['light_ch'] else
                                'table' if subject_row(fid) is not None else '&ndash; (bounds)')
                p['lights_avail'] = lights_avail
                ev_src = 'txt' if txt else 'csv' if csv else 'xml' if xml else '&ndash;'
                p['evt_avail'] = ev_src
                rows.append({'file_id': fid, 'subtree': str(p['subtree']),
                             'start': header['start_dt'].strftime('%Y-%m-%d %H:%M:%S') if header else '?',
                             'duration_min': round(header['duration_s'] / 60, 1) if header else np.nan,
                             'epochs_edf': n_edf_ep, 'epochs_hypno': n_hyp if n_hyp is not None else 'NO',
                             'lights': lights_avail, 'events': ev_src,
                             'spo2': p['spo2_ch'] or '&ndash;', 'processed': 'yes' if done else ''})
                parts.append(p)

            S.update({'parts': parts, 'data_root': data_root, 'out_root': out_root, 'data_dir': data_dir,
                      'reports_dir': reports_dir, 'custom_stages': custom, 'event_remap': remap,
                      'remap_note': remap_note})
            longest = max((len(str(p['out_data'] / f'{p["file_id"]}_sleep_metrics_checks.tsv'))
                           for p in parts), default=0)
            if longest > 245:
                warn.append(f'the longest output path is {longest} characters &mdash; close to the '
                            f'260-character Windows limit; move the dataset to a shorter path if files '
                            f'fail to be written.')
            if remap is None:
                warn.append(remap_note)
            display(HTML(f'<b>{len(parts)} recordings</b> &nbsp;|&nbsp; <b>{n_done} / {len(parts)}</b> '
                         f'already processed &nbsp;|&nbsp; Light channel in {n_light}, SpO2 in {n_spo2} '
                         f'&nbsp;|&nbsp; <small>{remap_note}</small>'))
            display(HTML(pd.DataFrame(rows).to_html(index=False, border=0, escape=False)))
            for w in warn:
                display(HTML(f'<span style="color:#ef6c00">&#9888; {w}</span>'))
            # pre-fill the Section-2 channel fields with what the files carry
            lights_names = sorted({p['light_ch'] for p in parts if p['light_ch']})
            spo2_names = sorted({p['spo2_ch'] for p in parts if p['spo2_ch']})
            txt_light_ch.value = lights_names[0] if lights_names else ''
            txt_spo2_ch.value = spo2_names[0] if spo2_names else ''
            build_participant_list()
            lbl_scan.value = (f'<span style="color:#2e7d32">Scan done &mdash; {len(parts)} recordings, '
                              f'{n_done} already processed.</span>')
        except Exception as exc:
            lbl_scan.value = f'<span style="color:#c62828">Scan failed: {exc}</span>'
            display(HTML(f'<pre style="color:#c62828">{exc}</pre>'))


fc_data.register_callback(_on_data)
fc_subj.register_callback(_on_subj)
btn_scan.on_click(scan_folder)

display(widgets.VBox([fc_data, fc_out, fc_remap, fc_subj, dd_join_col,
                      txt_hypno_suffix, lbl_hypno_suffix, txt_summary_suffix, lbl_summary_suffix,
                      txt_evt_txt_suffix, lbl_evt_txt_suffix, txt_evt_csv_suffix, lbl_evt_csv_suffix,
                      txt_custom, widgets.HBox([btn_scan, lbl_scan]), out_scan]))

## 2 — Parameters

**Lights-off / lights-on** decide the time-in-bed window. `auto` tries, in order, the Profusion summary
txt (`Luminosité` column, 1 = ON / 0 = OFF), the EDF `Light` channel, the two columns of the participant
table (clock time `HH:MM` or epoch index), and finally the recording start / end (always with a
warning). Force one source to check the others against it. The **sleep-onset rule** anchors SOL, SPT and
every latency; the default follows the definition file (first non-W epoch).

**ODI** and **T90** are opt-in: ODI needs the desaturation depth (read from the `.edf.XML` when the
events come from the txt/csv export), T90 reads the SpO2 signal of every recording — both make the run
slower.

In [ ]:
# =============================================================================
# Section 2 - parameters (lights, sleep onset, events, ODI, T90)
# =============================================================================
def _w(px):
    """A FRESH width Layout (never share a Layout object between widgets)."""
    return widgets.Layout(width=f'{px}px')


def _ind(px=14):
    return widgets.Layout(margin=f'0 0 0 {px}px')


def _desc(text, indent=14):
    return widgets.HTML(f'<div style="line-height:1.2;margin-left:{indent}px">'
                        f'<small style="color:#555">{text}</small></div>')


def _head(text):
    return widgets.HTML(f'<b style="font-size:110%">{text}</b>')


def _reveal(checkbox, *boxes):
    """Show/hide widgets from `checkbox`; the initial display is derived from the CURRENT value
    (an observe handler only fires on a change, so a pre-ticked box would stay hidden)."""
    for box in boxes:
        box.layout.display = '' if checkbox.value else 'none'

    def _on(change, boxes=boxes):
        for box in boxes:
            box.layout.display = '' if change['new'] else 'none'

    checkbox.observe(_on, names='value')


# ---------------------------------------------------------------------------
# 2a. Epoch length and lights
# ---------------------------------------------------------------------------
txt_epoch_s = widgets.IntText(description='Epoch length (s):', value=DEFAULT_EPOCH_S, layout=_w(220), style=_LBL)

LIGHTS_OPTIONS = [('auto (summary txt -> Light channel -> participant table -> recording bounds)', 'auto'),
                  ('summary txt (Profusion export)', 'summary_txt'),
                  ('EDF Light channel', 'light_channel'),
                  ('participant table columns', 'participant_table'),
                  ('recording start / end', 'recording_bounds')]
dd_lights_source = widgets.Dropdown(description='Lights-off/on source:', options=LIGHTS_OPTIONS,
                                    value='auto', layout=_w(620), style=_LBL)
txt_lights_col = widgets.Text(description='Summary txt column:', value='Luminosité', layout=_w(300), style=_LBL)
txt_light_ch = widgets.Text(description='Light channel:', value='', placeholder='auto (Light / Lights)',
                            layout=_w(300), style=_LBL)
txt_lights_off_col = widgets.Text(description='Table column lights-off:', value='lights_off',
                                  layout=_w(300), style=_LBL)
txt_lights_on_col = widgets.Text(description='Table column lights-on:', value='lights_on',
                                 layout=_w(300), style=_LBL)

box_lights = widgets.VBox([
    _head('Time in bed'),
    _desc('The hypnogram is assumed to start at the recording start, one label per epoch.'),
    widgets.HBox([txt_epoch_s], layout=_ind(14)),
    _desc('Lights-off / lights-on: each source is tried in the order shown; the first usable one wins '
          'and is recorded as <code>lights_source</code>. Recording start / end is the last resort and '
          'is always flagged. Summary txt and Light channel: 1 = lights ON, 0 = OFF; lights-off = first '
          'epoch after the last ON epoch before the first sleep epoch, lights-on = first ON epoch after '
          'the last sleep epoch.'),
    widgets.HBox([dd_lights_source], layout=_ind(14)),
    widgets.HBox([txt_lights_col, txt_light_ch], layout=_ind(14)),
    _desc('Participant table: clock time <code>HH:MM</code> / <code>HH:MM:SS</code> (rolled to the next '
          'day when after midnight) or a bare epoch index; a blank cell falls back to the recording bound.'),
    widgets.HBox([txt_lights_off_col, txt_lights_on_col], layout=_ind(14)),
])

# ---------------------------------------------------------------------------
# 2b. Sleep onset
# ---------------------------------------------------------------------------
dd_onset_rule = widgets.Dropdown(description='Sleep onset =', options=ONSET_RULES, value=ONSET_RULES[0],
                                 layout=_w(420), style=_LBL)
box_onset = widgets.VBox([
    _head('Sleep onset'),
    _desc('Anchor of SOL, SPT, TST and every stage latency. <i>first non-W epoch</i> is the definition '
          'of the formulas file (AASM); <i>first N2 epoch</i> and <i>3 consecutive sleep epochs</i> are '
          'common clinical variants. Under a variant the sleep epochs before the onset are reported but '
          'not counted.'),
    widgets.HBox([dd_onset_rule], layout=_ind(14)),
])

# ---------------------------------------------------------------------------
# 2c. Events, ODI, T90
# ---------------------------------------------------------------------------
cb_events = widgets.Checkbox(value=True, indent=False, layout=_w(620),
                             description='Event indices from the scored events (AHI + OAI/CAI/MAI/HI, PLM index, arousal index)')
cb_odi = widgets.Checkbox(value=False, indent=False, layout=_w(620),
                          description='ODI - oxygen desaturation index (slower: re-reads the .edf.XML for the depths)')
txt_desat_thr = widgets.FloatText(description='Desaturation depth >= (%):', value=3.0, layout=_w(260), style=_LBL)
box_odi = widgets.HBox([txt_desat_thr], layout=_ind(28))
cb_t90 = widgets.Checkbox(value=False, indent=False, layout=_w(620),
                          description='T90 - time with SpO2 below threshold (slower: reads the SpO2 signal of every recording)')
txt_spo2_ch = widgets.Text(description='SpO2 channel:', value='', placeholder='auto (SpO2 / SaO2)',
                           layout=_w(260), style=_LBL)
txt_spo2_thr = widgets.FloatText(description='SpO2 threshold (%):', value=90.0, layout=_w(220), style=_LBL)
txt_spo2_floor = widgets.FloatText(description='Artefact floor (%):', value=50.0, layout=_w(220), style=_LBL)
box_t90 = widgets.HBox([txt_spo2_ch, txt_spo2_thr, txt_spo2_floor], layout=_ind(28))
_reveal(cb_odi, box_odi)
_reveal(cb_t90, box_t90)

box_events = widgets.VBox([
    _head('Scored events and oxygen'),
    _desc('Events are counted between lights-off and lights-on (<code>EVENT_COUNT_WINDOW</code>) and '
          'divided by TST in hours. Labels come from <code>event_remap.json</code> (tool 4): '
          '<code>apnea_obstructive</code> / <code>apnea_central</code> / <code>apnea_mixed</code>, '
          '<code>hypopnea*</code>, <code>plm</code>, <code>arousal*</code>, <code>spo2_desaturation</code>. '
          'A label absent from the remap is NOT counted and is reported.'),
    widgets.VBox([cb_events], layout=_ind(14)),
    widgets.VBox([cb_odi], layout=_ind(14)),
    _desc('Report which threshold was used (3 % = current AASM recommendation, 4 % in older studies); '
          'without any depth information every scored desaturation is counted.', indent=28),
    box_odi,
    widgets.VBox([cb_t90], layout=_ind(14)),
    _desc('Samples <= 0 or below the artefact floor (sensor off, motion) are discarded and their '
          'percentage reported; T90 is given in minutes and as % of TST.', indent=28),
    box_t90,
])

lbl_params = widgets.HTML()


def get_params():
    """Read every Section-2 widget into one config dict; returns (cfg, errors)."""
    errors = []
    epoch_s = int(txt_epoch_s.value)
    if epoch_s <= 0:
        errors.append('the epoch length must be positive')
    if cb_odi.value and float(txt_desat_thr.value) < 0:
        errors.append('the desaturation threshold must be >= 0')
    if cb_t90.value and not (0 < float(txt_spo2_thr.value) <= 100):
        errors.append('the SpO2 threshold must be in (0, 100]')
    if cb_t90.value and not (0 <= float(txt_spo2_floor.value) < float(txt_spo2_thr.value)):
        errors.append('the artefact floor must be below the SpO2 threshold')
    cfg = dict(epoch_s=epoch_s, lights_source=dd_lights_source.value,
               lights_col=txt_lights_col.value.strip() or 'Luminosité',
               light_ch=txt_light_ch.value.strip(),
               lights_off_col=txt_lights_off_col.value.strip() or 'lights_off',
               lights_on_col=txt_lights_on_col.value.strip() or 'lights_on',
               onset_rule=dd_onset_rule.value, events=bool(cb_events.value),
               odi=bool(cb_odi.value), odi_thr=float(txt_desat_thr.value),
               t90=bool(cb_t90.value), spo2_ch=txt_spo2_ch.value.strip(),
               spo2_thr=float(txt_spo2_thr.value), spo2_floor=float(txt_spo2_floor.value),
               other_stage_policy=OTHER_STAGE_POLICY, event_count_window=EVENT_COUNT_WINDOW)
    return cfg, errors


display(widgets.VBox([box_lights, box_onset, box_events, lbl_params]))

## 3 — Participants

Tick the recordings to process. A recording counts as **already processed** when its
`{file_id}_sleep_metrics.tsv` is on disk (delete it to force a reprocess); the global tables are always
rebuilt from every per-recording file present, so partial runs accumulate.

In [ ]:
# =============================================================================
# Section 3 - participant selection
# =============================================================================
part_box = widgets.VBox([])
part_checkboxes = {}
cb_skip = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width='420px'),
                           description='Skip already processed participants')
btn_all = widgets.Button(description='Select all', layout=widgets.Layout(width='120px'))
btn_none = widgets.Button(description='None', layout=widgets.Layout(width='90px'))
lbl_part = widgets.HTML('<i>Run the scan in Section 1 first.</i>')


def build_participant_list():
    """One checkbox per recording, ticked by default; what was found next to it is annotated."""
    part_checkboxes.clear()
    rows = []
    for p in S.get('parts', []):
        tag = ' <span style="color:#2e7d32">(already processed)</span>' if p.get('done') else ''
        hyp = ('hypno &#10003;' if p.get('hypno') is not None
               else '<span style="color:#c62828">hypno &#10007;</span>')
        cb = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width='300px'),
                              description=p['file_id'])
        part_checkboxes[p['file_id']] = cb
        rows.append(widgets.HBox([cb, widgets.HTML(
            f'<small>{hyp} &middot; events {p.get("evt_avail", "&ndash;")} &middot; lights '
            f'{p.get("lights_avail", "&ndash;")} &middot; SpO2 {p.get("spo2_ch") or "&ndash;"}{tag}</small>')]))
    part_box.children = tuple(rows)
    n_done = sum(1 for p in S.get('parts', []) if p.get('done'))
    lbl_part.value = (f'<b>{len(part_checkboxes)}</b> recordings &mdash; '
                      f'<b>{n_done} / {len(part_checkboxes)}</b> already processed.')


def selected_participants():
    """The recordings to run, honouring the checkboxes and the skip option."""
    out = []
    for p in S.get('parts', []):
        cb = part_checkboxes.get(p['file_id'])
        if cb is None or not cb.value:
            continue
        if cb_skip.value and p.get('done'):
            continue
        out.append(p)
    return out


btn_all.on_click(lambda b: [setattr(cb, 'value', True) for cb in part_checkboxes.values()])
btn_none.on_click(lambda b: [setattr(cb, 'value', False) for cb in part_checkboxes.values()])

display(widgets.VBox([widgets.HBox([btn_all, btn_none, cb_skip]), lbl_part, part_box]))

## 4 — Run

For each recording: read the EDF header, load and check the hypnogram, resolve lights-off/on, compute
the macrostructure, count the scored events (and optionally ODI / T90), run the consistency checks,
then write `{file_id}_sleep_metrics_checks.tsv` and `{file_id}_sleep_metrics.tsv`. A recording that
fails (no hypnogram, length mismatch) is listed in `sleep_metrics_failed.tsv` and never stops the run.
The database tables, the workbook and the report are then rebuilt from every per-recording file on disk.

In [ ]:
# =============================================================================
# Section 4 - run: per recording, then the database-level outputs
# =============================================================================
EXCEL_MAX_ROWS = 1048575

btn_run = widgets.Button(description='Run', button_style='success', icon='play')
lbl_run = widgets.HTML()
progress_part = widgets.IntProgress(value=0, min=0, max=1, description='Participants:',
                                    layout=widgets.Layout(width='620px'), style=_LBL)
progress_step = widgets.IntProgress(value=0, min=0, max=1, bar_style='info', description='Pipeline:',
                                    layout=widgets.Layout(width='620px'), style=_LBL)
lbl_phase = widgets.HTML()
out_run = widgets.Output()

_STEP_LEGEND = widgets.HTML(
    '<div style="width:620px;font-size:85%;color:#555;display:flex;text-align:center">'
    + ''.join(f'<div style="flex:{c};border-right:1px solid #ccc">{n}</div>'
              for n, c in [('Header', COST_HEADER), ('Hypnogram', COST_HYPNO), ('Lights', COST_LIGHTS),
                           ('Events', COST_EVENTS), ('SpO2', COST_SPO2), ('Write', COST_WRITE)])
    + '</div>')


def _write_tsv(df, path):
    """Write a table, creating the folder. Empty tables are skipped."""
    if df is None or len(df) == 0:
        return None
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep='\t', index=False)
    return path


def _join_subject_info(df):
    """Left-join the optional participant table on file_id (normcase-insensitive on both sides)."""
    info = S.get('subj_info')
    col = dd_join_col.value
    if df is None or len(df) == 0 or info is None or not col or col not in info.columns:
        return df
    try:
        left = df.copy()
        left['_key'] = left['file_id'].astype(str).map(norm)
        right = info.copy()
        right['_key'] = right[col].astype(str).str.strip().map(norm)
        right = right.drop_duplicates('_key')
        right = right.drop(columns=[c for c in right.columns if c in left.columns and c != '_key'])
        return left.merge(right, on='_key', how='left').drop(columns=['_key'])
    except Exception:
        return df


def _clock(start_dt, epoch_idx, epoch_s):
    return (start_dt + datetime.timedelta(seconds=int(epoch_idx) * epoch_s)).strftime('%H:%M:%S')


def process_participant(p, cfg, log):
    """Full pipeline for one recording. Returns a one-line summary; a fatal step raises."""
    fid = p['file_id']
    steps = (COST_HEADER + COST_HYPNO + COST_LIGHTS + COST_WRITE
             + (COST_EVENTS if (cfg['events'] or cfg['odi']) else 0) + (COST_SPO2 if cfg['t90'] else 0))
    progress_step.max = steps
    progress_step.value = 0

    def phase(name, add):
        lbl_phase.value = f'<small>{fid} &mdash; {name}</small>'
        progress_step.value = min(progress_step.value + add, steps)

    checks = []
    epoch_s = cfg['epoch_s']
    # --- header + hypnogram -------------------------------------------------
    phase('reading the EDF header', 0)
    header = p.get('header') or read_edf_header_info(p['edf'])
    phase('loading the hypnogram', COST_HEADER)
    if p.get('hypno') is None:
        raise FileNotFoundError(f'no hypnogram {fid}{txt_hypno_suffix.value} next to the EDF')
    stages = load_hypnogram(p['hypno'])
    stages, chk = reconcile_hypno_length(stages, header['duration_s'], epoch_s)
    checks.append(chk)
    if chk['status'] == 'fail':
        raise ValueError(chk['detail'])
    custom = S.get('custom_stages', [])
    cls = classify_stages(stages, custom)
    n_hyp = len(stages)

    # --- lights ---------------------------------------------------------------
    phase('resolving lights-off / lights-on', COST_HYPNO)
    light_ch = cfg['light_ch'] if cfg['light_ch'] and cfg['light_ch'] in header['ch_names'] else p.get('light_ch')
    lights = resolve_lights(cfg, {**p, 'light_ch': light_ch}, header, n_hyp, cls['sleep_mask'],
                            subject_row(fid))
    checks += lights['checks']
    off_idx, on_idx = lights['off_idx'], lights['on_idx']

    # --- macrostructure -------------------------------------------------------
    metrics, mchecks, extras = compute_macrostructure(stages, cls, off_idx, on_idx, epoch_s, cfg['onset_rule'])
    checks += mchecks
    tst = metrics.get('tst_min', np.nan)

    # --- events ---------------------------------------------------------------
    phase('counting the scored events', COST_LIGHTS)
    event_info = {'enabled': cfg['events'] or cfg['odi'], 'source': None, 'n_total': 0,
                  'n_outside': 0, 'n_bad_onset': 0, 'n_unmapped': 0}
    odi_rule = ''
    if cfg['events'] or cfg['odi']:
        remap = S.get('event_remap')
        events_df, source = load_events(p['edf'], txt_evt_txt_suffix.value, txt_evt_csv_suffix.value)
        event_info['source'] = source
        if events_df is not None:
            event_info['n_total'] = int(len(events_df))
            win_df, n_out, n_bad = select_events_in_window(events_df, cls, off_idx, on_idx, epoch_s,
                                                           EVENT_COUNT_WINDOW)
            event_info.update({'n_outside': n_out, 'n_bad_onset': n_bad})
            if remap is None:
                checks.append({'check': 'event_remap', 'status': 'warning',
                               'detail': 'no event_remap.json - the scored events could not be '
                                         'classified (run tool 4); event indices are NaN'})
            else:
                if cfg['events']:
                    em, ec, n_unm = compute_event_indices(win_df, remap, tst)
                    metrics.update(em)
                    checks += ec
                    event_info['n_unmapped'] = n_unm
                if cfg['odi']:
                    om, oc, odi_rule = compute_odi(win_df, remap, tst, cfg['odi_thr'], p.get('evt_xml'),
                                                   cls, off_idx, on_idx, epoch_s, EVENT_COUNT_WINDOW)
                    metrics.update(om)
                    checks += oc

    # --- SpO2 -----------------------------------------------------------------
    spo2_ch = ''
    if cfg['t90']:
        phase('reading the SpO2 signal', COST_EVENTS if (cfg['events'] or cfg['odi']) else 0)
        spo2_ch = cfg['spo2_ch'] if cfg['spo2_ch'] and cfg['spo2_ch'] in header['ch_names'] else p.get('spo2_ch')
        if not spo2_ch:
            checks.append({'check': 'spo2_channel', 'status': 'warning',
                           'detail': 'no SpO2 channel found - T90 not computed'})
            spo2_ch = ''
        else:
            try:
                x, sf, rescaled = load_spo2_signal(p['edf'], spo2_ch)
                if rescaled:
                    checks.append({'check': 'spo2_units', 'status': 'warning',
                                   'detail': 'SpO2 stored as a fraction (max <= 1) - rescaled to %'})
                tm, tc = compute_t90(x, sf, off_idx, on_idx, epoch_s, tst, cfg['spo2_thr'], cfg['spo2_floor'])
                metrics.update(tm)
                checks += tc
                del x
            except Exception as exc:
                checks.append({'check': 'spo2_channel', 'status': 'warning',
                               'detail': f'SpO2 signal unreadable ({exc}) - T90 not computed'})

    # --- checks, provenance, write --------------------------------------------
    phase('checks and writing', (COST_SPO2 if cfg['t90'] else 0)
          + (0 if cfg['t90'] else (COST_EVENTS if (cfg['events'] or cfg['odi']) else 0)))
    checks += run_checks(metrics, extras, lights, n_hyp, cls, event_info)
    n_warn = sum(1 for c in checks if c['status'] in ('warning', 'fail'))
    provenance = {
        'lights_source': lights['source'],
        'lights_off_clock': _clock(header['start_dt'], off_idx, epoch_s),
        'lights_on_clock': _clock(header['start_dt'], on_idx, epoch_s),
        'lights_off_epoch': off_idx, 'lights_on_epoch': on_idx,
        'sleep_onset_rule': cfg['onset_rule'],
        'sleep_onset_epoch': extras['onset_idx_abs'] if extras['onset_idx_abs'] is not None else '',
        'other_stage_policy': OTHER_STAGE_POLICY, 'event_count_window': EVENT_COUNT_WINDOW,
        'epoch_length_s': epoch_s, 'n_epochs_hypno': n_hyp, 'n_epochs_in_bed': extras['n_epochs_in_bed'],
        'n_mt_custom': int(cls['other_mask'].sum()),
        'event_source': event_info['source'] or '', 'n_events_total': event_info['n_total'],
        'n_unmapped_events': event_info['n_unmapped'], 'n_events_outside_window': event_info['n_outside'],
        'has_spo2': bool(spo2_ch), 'odi_threshold_pct': cfg['odi_thr'] if cfg['odi'] else '',
        'odi_rule': odi_rule, 'spo2_channel': spo2_ch, 'light_channel': lights.get('light_channel', ''),
        'recording_start': header['start_dt'].strftime('%Y-%m-%d %H:%M:%S'),
        'edf_duration_min': round(header['duration_s'] / 60, 2), 'n_warnings': n_warn}
    out_data = p['out_data']
    _write_tsv(checks_df(fid, checks), out_data / f'{fid}_sleep_metrics_checks.tsv')
    _write_tsv(metrics_long_df(fid, metrics, provenance), out_data / f'{fid}_sleep_metrics.tsv')
    progress_step.value = steps

    def fmt(k):
        v = metrics.get(k, np.nan)
        return f'{v:.1f}' if v is not None and np.isfinite(v) else 'NaN'
    summary = (f'TIB {fmt("tib_min")} &middot; SOL {fmt("sol_min")} &middot; TST {fmt("tst_min")} &middot; '
               f'WASO {fmt("waso_min")} &middot; SE {fmt("se_pct")} % &middot; REM lat {fmt("lat_rem_min")}'
               + (f' &middot; AHI {fmt("ahi")}' if cfg['events'] else '')
               + (f' &middot; ODI {fmt("odi")}' if cfg['odi'] else '')
               + (f' &middot; T90 {fmt("t90_min")} min' if cfg['t90'] else '')
               + f' &middot; lights: {lights["source"]}')
    for c in checks:
        if c['status'] in ('warning', 'fail'):
            log(f'  <span style="color:{STATUS_COLORS[c["status"]]}">&#9888; {c["check"]}: {c["detail"]}</span>')
    return summary


def rebuild_globals(cfg, log):
    """Rebuild the database tables by globbing the per-recording files from disk."""
    data_dir, reports_dir = S['data_dir'], S['reports_dir']
    reports_dir.mkdir(parents=True, exist_ok=True)
    frames, cframes = [], []
    for f in sorted(data_dir.rglob('*_sleep_metrics.tsv')):
        if norm(f.name).startswith(norm('global_')):
            continue
        try:
            # file_id forced to str: a numeric id ("73") would otherwise come back as an int
            frames.append(pd.read_csv(f, sep='\t', dtype={'file_id': str, 'value': str}, keep_default_na=False))
        except Exception as exc:
            log(f'&#9888; could not read {f.name}: {exc}')
    for f in sorted(data_dir.rglob('*_sleep_metrics_checks.tsv')):
        try:
            cframes.append(pd.read_csv(f, sep='\t', dtype={'file_id': str}))
        except Exception as exc:
            log(f'&#9888; could not read {f.name}: {exc}')
    long_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    wide = long_to_wide(long_df)
    wide = _join_subject_info(wide)
    checks = pd.concat(cframes, ignore_index=True) if cframes else pd.DataFrame(
        columns=['file_id', 'check', 'status', 'detail'])
    _write_tsv(wide, reports_dir / 'global_sleep_metrics.tsv')
    _write_tsv(checks, reports_dir / 'global_sleep_metrics_checks.tsv')
    return {'wide': wide, 'checks': checks}


def glossary_df():
    rows = []
    for key, label, unit, group, definition in METRICS:
        rng = REFERENCE_RANGES.get(key)
        rows.append({'metric': key, 'label': label, 'unit': unit, 'group': group, 'definition': definition,
                     'indicative_low': rng[0] if rng else np.nan, 'indicative_high': rng[1] if rng else np.nan,
                     'note': rng[2] if rng else ''})
    return pd.DataFrame(rows)


def write_excel(tables, cfg, failed, path, log):
    """One workbook, one sheet per table (the TSV files stay the source of truth)."""
    if not HAS_OPENPYXL:
        log('&#9888; openpyxl is missing - the .xlsx workbook was not written (the TSV tables are complete)')
        return None
    par = pd.DataFrame({'parameter': list(cfg.keys()), 'value': [str(v) for v in cfg.values()]})
    sheets = {'sleep_metrics': tables.get('wide'), 'checks': tables.get('checks'),
              'glossary': glossary_df(), 'parameters': par,
              'failed': pd.DataFrame(failed, columns=['file_id', 'reason']) if failed else pd.DataFrame()}
    try:
        with pd.ExcelWriter(path, engine='openpyxl') as xl:
            for name, df in sheets.items():
                if df is None or len(df) == 0:
                    pd.DataFrame({'note': ['(empty)']}).to_excel(xl, sheet_name=name, index=False)
                    continue
                if len(df) > EXCEL_MAX_ROWS:
                    log(f'&#9888; sheet "{name}" truncated to {EXCEL_MAX_ROWS} rows - use the TSV')
                    df = df.head(EXCEL_MAX_ROWS)
                df.to_excel(xl, sheet_name=name, index=False)
        return path
    except Exception as exc:
        log(f'&#9888; could not write the workbook: {exc}')
        return None


def build_database_report(tables, cfg, path, comparison=None):
    """Database-level report: summary, lights overview, metric distributions with the indicative
    ranges, checks, participant table, optional comparison, glossary, and "where to look"."""
    wide, checks = tables.get('wide'), tables.get('checks')
    n_part = len(wide) if wide is not None else 0
    src_counts = wide['lights_source'].value_counts().to_dict() if n_part else {}
    n_warn = int((pd.to_numeric(wide['n_warnings'], errors='coerce') > 0).sum()) if n_part else 0
    n_fail = checks.loc[checks['status'] == 'fail', 'file_id'].nunique() if checks is not None and len(checks) else 0
    items = [
        ('html', 'Summary', params_html({
            'Participants in the table': n_part,
            'Lights-off/on source': ', '.join(f'{k}: {v}' for k, v in src_counts.items()) or '-',
            'Sleep-onset rule': cfg.get('onset_rule', ''),
            'Epoch length (s)': cfg.get('epoch_s', ''),
            'MT / custom stage policy': OTHER_STAGE_POLICY,
            'Event counting window': EVENT_COUNT_WINDOW,
            'Event remap': S.get('remap_note', ''),
            'ODI': f"on, depth >= {cfg.get('odi_thr')} %" if cfg.get('odi') else 'off',
            'T90': f"on, SpO2 < {cfg.get('spo2_thr')} % (floor {cfg.get('spo2_floor')} %)" if cfg.get('t90') else 'off',
            'Participants with warnings / failures': f'{n_warn} / {n_fail}',
        })),
        ('fig', 'Time in bed and sleep period', plot_lights_overview(wide) if n_part else None),
        ('fig', 'Sleep macrostructure', plot_metric_distributions(wide, ('macrostructure',),
                                                                  'Sleep macrostructure') if n_part else None),
        ('fig', 'Event indices', plot_metric_distributions(wide, ('respiratory', 'plm', 'arousal', 'odi'),
                                                           'Event indices') if n_part else None),
        ('fig', 'Oxygen saturation', plot_metric_distributions(wide, ('spo2',), 'Oxygen saturation') if n_part else None),
        ('html', 'Consistency checks', checks_summary_html(checks)),
        ('html', 'Participants', participants_html(wide)),
    ]
    if comparison is not None and len(comparison):
        mism = comparison[comparison['status'] == 'mismatch']
        counts = comparison['status'].value_counts().to_dict()
        html = ('<p>' + ', '.join(f'<b>{k}</b>: {v}' for k, v in counts.items()) + '</p>'
                + (mism.round(3).to_html(index=False, border=0) if len(mism) else '<p>No mismatch.</p>'))
        items += [('fig', 'Comparison with the reference table', plot_comparison(comparison)),
                  ('html', 'Comparison - mismatches', html)]
    items += [('html', 'Metric glossary', glossary_html()),
              ('html', 'Where to look', guidance_html(cfg, S['data_dir'], S['reports_dir'],
                                                      has_subject_info=S.get('subj_info') is not None,
                                                      has_comparison=comparison is not None))]
    save_report_html(path, 'Sleep macrostructure - database report', items)
    close_figs(items)
    return path


def finalize_outputs(cfg, failed, log, comparison=None):
    """Rebuild the database tables, the workbook and the report (all non-fatal)."""
    reports_dir = S['reports_dir']
    tables, xlsx = {}, None
    try:
        tables = rebuild_globals(cfg, log)
        # always rewritten (header only when nothing failed) so a stale list never survives a clean run
        pd.DataFrame(failed, columns=['file_id', 'reason']).to_csv(
            reports_dir / 'sleep_metrics_failed.tsv', sep='	', index=False)
    except Exception as exc:
        log(f'&#9888; could not rebuild the database tables: {exc}')
    try:
        xlsx = write_excel(tables, cfg, failed, reports_dir / 'sleep_metrics_database.xlsx', log)
    except Exception as exc:
        log(f'&#9888; could not write the workbook: {exc}')
    try:
        build_database_report(tables, cfg, reports_dir / 'sleep_metrics_database_report.html', comparison)
    except Exception as exc:
        log(f'&#9888; could not write the database report: {exc}')
    return tables, xlsx


def run(_=None):
    """Section-4 entry point: loop over the selected recordings, then rebuild the database outputs."""
    with out_run:
        clear_output()

        def log(msg):
            display(HTML(msg))

        try:
            if not S.get('parts'):
                lbl_run.value = '<span style="color:#c62828">Run the scan in Section 1 first.</span>'
                return
            cfg, errors = get_params()
            if errors:
                lbl_run.value = ('<span style="color:#c62828">' +
                                 '<br>'.join('&#10007; ' + e for e in errors) + '</span>')
                return
            S['cfg'] = cfg
            todo = selected_participants()
            n_skipped = sum(1 for p in S['parts']
                            if part_checkboxes.get(p['file_id']) and part_checkboxes[p['file_id']].value
                            and p.get('done')) if cb_skip.value else 0
            if not todo:
                lbl_run.value = ('<span style="color:#ef6c00">Nothing to do &mdash; every selected '
                                 'recording is already processed (untick Skip to redo them). The database '
                                 'outputs are rebuilt anyway.</span>')
            progress_part.max = max(len(todo), 1)
            progress_part.value = 0
            if todo:
                lbl_run.value = f'<i>Running {len(todo)} recordings...</i>'

            done, failed = [], []
            for i, p in enumerate(todo, 1):
                progress_part.value = i - 1
                progress_part.description = f'Participants: {i}/{len(todo)}'
                try:
                    summary = process_participant(p, cfg, log)
                    p['done'] = True
                    done.append(p['file_id'])
                    log(f'<b>{p["file_id"]}</b> &mdash; <small>{summary}</small>')
                except Exception as exc:
                    failed.append((p['file_id'], f'{type(exc).__name__}: {exc}'))
                    log(f'<b>{p["file_id"]}</b> <span style="color:#c62828">&#10007; failed: {exc}</span>')
                progress_part.value = i

            lbl_phase.value = '<small>rebuilding the database tables...</small>'
            tables, xlsx = finalize_outputs(cfg, failed, log)
            lbl_phase.value = ''
            reports_dir = S['reports_dir']
            wide = tables.get('wide')
            n_part = len(wide) if wide is not None else 0
            log(f'<hr><b>{len(done)} processed</b>, {len(failed)} failed, {n_skipped} skipped &mdash; '
                f'the database table now covers <b>{n_part}</b> recordings.')
            log('<div style="background:#e8f5e9;border-left:4px solid #2e7d32;padding:8px 12px;margin:8px 0">'
                f'<b>Start here:</b> <code>{reports_dir / "global_sleep_metrics.tsv"}</code> (one row per '
                f'recording) and <code>{reports_dir / "sleep_metrics_database_report.html"}</code>, whose '
                'closing <i>Where to look</i> section maps every file this run produced.</div>')
            log(f'Data: <code>{S["data_dir"]}</code><br>Reports: <code>{reports_dir}</code>'
                + (f'<br>Workbook: <code>{xlsx}</code>' if xlsx else ''))
            if n_part:
                display(HTML(participants_html(wide)))
            if todo:
                lbl_run.value = (f'<span style="color:#2e7d32">Done &mdash; {len(done)} processed, '
                                 f'{len(failed)} failed.</span>')
            build_participant_list()
        except Exception as exc:
            lbl_run.value = f'<span style="color:#c62828">Run failed: {exc}</span>'
            display(HTML(f'<pre style="color:#c62828">{exc}</pre>'))


btn_run.on_click(run)
display(widgets.VBox([widgets.HBox([btn_run, lbl_run]), progress_part,
                      widgets.VBox([progress_step, _STEP_LEGEND]), lbl_phase, out_run]))

## 5 — Compare with a reference table (optional)

The same metrics are printed on the clinical PSG reports. Load a table with **one row per participant**
(`.csv` / `.tsv` / `.xlsx`): pick the id column, check the metric → column mapping (auto-guessed from
the column names, `(none)` = not compared) and the tolerance of each metric (default 1 min / 1 %-point /
1 event/h, exact for counts), then **Compare**. The result is written to
`reports_features_macrostructure/comparison_with_reference.tsv` and added to the database report.

A **systematic offset** on one metric points to a definition difference (lights-off/on source, sleep-onset
rule, MT handling, events counted over TIB vs the whole recording); an **isolated mismatch** points to a
scoring or file problem for that participant.

In [ ]:
# =============================================================================
# Section 5 - comparison with an external reference table
# =============================================================================
fc_ref = FileChooser(os.getcwd())
fc_ref.filter_pattern = ['*.csv', '*.tsv', '*.xlsx']
fc_ref.title = '<b>Reference table</b> (one row per participant; e.g. the values of the clinical reports):'
dd_ref_id = widgets.Dropdown(description='Participant id column:', options=[], value=None,
                             layout=widgets.Layout(width='360px'), style=_LBL)
btn_guess = widgets.Button(description='Re-guess mapping', icon='magic', layout=widgets.Layout(width='160px'))
btn_compare = widgets.Button(description='Compare', button_style='success', icon='check')
lbl_compare = widgets.HTML('<i>Run Section 4 first, then load a reference table.</i>')
map_box = widgets.VBox([], layout=widgets.Layout(max_height='420px', overflow_y='auto',
                                                 border='1px solid #ddd', padding='4px'))
map_widgets = {}          # metric key -> (dropdown, tolerance)
out_compare = widgets.Output()


def _build_mapping_rows(ref_columns):
    """One row per metric: label, reference-column dropdown, tolerance field."""
    map_widgets.clear()
    guess = guess_metric_mapping(ref_columns)
    rows = [widgets.HTML('<small style="color:#555">metric &nbsp;|&nbsp; reference column &nbsp;|&nbsp; '
                         'tolerance (absolute, in the metric unit)</small>')]
    for key in METRIC_KEYS:
        dd = widgets.Dropdown(options=['(none)'] + [str(c) for c in ref_columns], value=guess.get(key, '(none)'),
                              layout=widgets.Layout(width='260px'))
        tol = widgets.FloatText(value=default_tolerance(key), layout=widgets.Layout(width='90px'))
        map_widgets[key] = (dd, tol)
        info = METRIC_INFO[key]
        rows.append(widgets.HBox([
            widgets.HTML(f'<div style="width:300px"><b>{info["label"]}</b> '
                         f'<small style="color:#777">({info["unit"]})</small></div>'), dd, tol],
            layout=widgets.Layout(overflow='hidden')))
    map_box.children = tuple(rows)
    n_guessed = sum(1 for k in METRIC_KEYS if guess.get(k, '(none)') != '(none)')
    return n_guessed


def _on_ref(chooser):
    try:
        df, err = load_subject_info(fc_ref.selected)
        if df is None:
            lbl_compare.value = f'<span style="color:#c62828">Reference table unreadable: {err}</span>'
            return
        S['ref_df'] = df
        dd_ref_id.options = list(df.columns)
        guess = [c for c in df.columns if norm(c) in ('file_id', 'fileid', 'sub_id', 'subid', 'id',
                                                       'participant', 'participant_id', 'subject', 'patient')]
        dd_ref_id.value = guess[0] if guess else list(df.columns)[0]
        n = _build_mapping_rows(df.columns)
        lbl_compare.value = (f'<i>Reference loaded: {len(df)} rows, {len(df.columns)} columns &mdash; '
                             f'{n} metric(s) matched automatically; check the mapping, then Compare.</i>')
    except Exception as exc:
        lbl_compare.value = f'<span style="color:#c62828">Reference error: {exc}</span>'


def _on_guess(_=None):
    if S.get('ref_df') is not None:
        n = _build_mapping_rows(S['ref_df'].columns)
        lbl_compare.value = f'<i>{n} metric(s) matched automatically.</i>'


def run_comparison(_=None):
    """Compare global_sleep_metrics.tsv (from disk) with the reference table, write the result,
    and rebuild the database report with the comparison section."""
    with out_compare:
        clear_output()

        def log(msg):
            display(HTML(msg))

        try:
            if not S.get('reports_dir'):
                lbl_compare.value = '<span style="color:#c62828">Run the scan (Section 1) and the run (Section 4) first.</span>'
                return
            gpath = S['reports_dir'] / 'global_sleep_metrics.tsv'
            if not gpath.exists():
                lbl_compare.value = f'<span style="color:#c62828">{gpath} not found - run Section 4 first.</span>'
                return
            ref = S.get('ref_df')
            if ref is None or not dd_ref_id.value:
                lbl_compare.value = '<span style="color:#c62828">Load a reference table first.</span>'
                return
            wide = pd.read_csv(gpath, sep='\t', dtype={'file_id': str})
            mapping = {k: dd.value for k, (dd, _) in map_widgets.items()}
            tol = {k: float(t.value) for k, (_, t) in map_widgets.items()}
            if all(v == '(none)' for v in mapping.values()):
                lbl_compare.value = '<span style="color:#c62828">No metric mapped to a reference column.</span>'
                return
            cmp_df = compare_with_reference(wide, ref, dd_ref_id.value, mapping, tol)
            _write_tsv(cmp_df, S['reports_dir'] / 'comparison_with_reference.tsv')
            S['comparison'] = cmp_df
            counts = cmp_df['status'].value_counts().to_dict()
            log('<b>Comparison:</b> ' + ', '.join(f'{k} {v}' for k, v in counts.items()))
            n_ref_only = len(set(ref[dd_ref_id.value].astype(str).str.strip().map(norm))
                             - set(wide['file_id'].astype(str).map(norm)))
            if n_ref_only:
                log(f'<span style="color:#ef6c00">&#9888; {n_ref_only} participant(s) of the reference '
                    f'table have no row in global_sleep_metrics.tsv (id mismatch or not processed).</span>')
            mism = cmp_df[cmp_df['status'] == 'mismatch']
            if len(mism):
                per_metric = mism.groupby('metric').agg(n=('file_id', 'count'), mean_diff=('diff', 'mean'))
                log('<b>Mismatches per metric</b> (a mean diff far from 0 = systematic definition difference):')
                display(HTML(per_metric.round(2).reset_index().to_html(index=False, border=0)))
                display(HTML(mism.round(3).to_html(index=False, border=0)))
            else:
                log('<span style="color:#2e7d32">No mismatch beyond the tolerances.</span>')
            cfg = S.get('cfg') or get_params()[0]
            try:
                tables = rebuild_globals(cfg, log)
                build_database_report(tables, cfg, S['reports_dir'] / 'sleep_metrics_database_report.html',
                                      comparison=cmp_df)
                log(f'Report updated: <code>{S["reports_dir"] / "sleep_metrics_database_report.html"}</code>'
                    f'<br>Table: <code>{S["reports_dir"] / "comparison_with_reference.tsv"}</code>')
            except Exception as exc:
                log(f'&#9888; comparison written but the report could not be rebuilt: {exc}')
            lbl_compare.value = (f'<span style="color:#2e7d32">Done &mdash; {counts.get("match", 0)} match, '
                                 f'{counts.get("mismatch", 0)} mismatch.</span>')
        except Exception as exc:
            lbl_compare.value = f'<span style="color:#c62828">Comparison failed: {exc}</span>'
            display(HTML(f'<pre style="color:#c62828">{exc}</pre>'))


fc_ref.register_callback(_on_ref)
btn_guess.on_click(_on_guess)
btn_compare.on_click(run_comparison)
display(widgets.VBox([fc_ref, widgets.HBox([dd_ref_id, btn_guess]), map_box,
                      widgets.HBox([btn_compare, lbl_compare]), out_compare]))

## Outputs — where to look

**Start with** `reports_features_macrostructure/global_sleep_metrics.tsv` — one row per recording, the
table for statistics — and the database report `sleep_metrics_database_report.html`, whose closing
*Where to look* section repeats this guide.

| File | What it holds | How to read it |
|---|---|---|
| `reports_features_macrostructure/global_sleep_metrics.tsv` | `file_id`, one column per metric (`tib_min`, `sol_min`, `spt_min`, `tst_min`, `waso_min`, `se_pct`, `sme_pct`, `lat_*_min`, `n1_pct` … `rem_pct`, `slow_sleep_*`, `ahi` / `oai` / `cai` / `mai` / `hi`, `plm_index`, `arousal_index`, `odi`, `t90_*`, `spo2_*`, plus the raw event counts `n_*`), then the provenance columns (`lights_source`, `lights_off_clock`, `lights_on_clock`, `sleep_onset_rule`, `n_mt_custom`, `event_source`, `n_unmapped_events`, `odi_rule`, `n_warnings` …) and, when a participant table was given, its columns joined on the right. Rebuilt from every per-recording file on disk at each run. | Units: minutes, %, events/h, counts (see the glossary). **Before trusting a value**: `lights_source = recording_bounds` means TIB is the whole recording (SOL and SE biased); `n_unmapped_events > 0` means events absent from `event_remap.json` were *not* counted in the indices (map them with tool 4, re-run with *Skip* unticked); `n_warnings > 0` → read the checks file. Group by any participant-info column for population comparisons. |
| `reports_features_macrostructure/global_sleep_metrics_checks.tsv` | Every consistency check of every recording: `check`, `status` (`ok` / `info` / `warning` / `fail`), `detail`. | A `fail` means a metric could not be computed or trusted (`no_sleep`, `hypno_vs_edf_length`); a `warning` flags a fallback or an unusual situation (`lights_fallback`, `no_lights_on`, `mt_custom_present`, `unmapped_events`, `spo2_artifact`, `odi_rule` …). |
| `reports_features_macrostructure/sleep_metrics_database.xlsx` | Sheets `sleep_metrics`, `checks`, `glossary` (definitions + indicative ranges), `parameters` (this run), `failed`. | Convenience copy; the TSV files are the source of truth. |
| `reports_features_macrostructure/sleep_metrics_database_report.html` | Time-in-bed overview per recording (coloured by lights source), distribution of every metric with the indicative normal range shaded, checks summary, participant table, comparison (if run), glossary, *Where to look*. | The entry point for a first look at a database. |
| `reports_features_macrostructure/sleep_metrics_failed.tsv` | Recordings that produced no metrics and why. | Missing hypnogram, hypnogram/EDF length mismatch > 1 epoch, unreadable EDF header. |
| `reports_features_macrostructure/comparison_with_reference.tsv` | One row per (recording, compared metric): `ours`, `reference`, `ref_column`, `diff`, `tol`, `status` (`match` / `mismatch` / `missing_ours` / `missing_reference` / `not_in_reference_table`). | Filter on `mismatch`; a mean `diff` far from 0 on one metric = definition difference, an isolated outlier = scoring or file problem. |
| `derivatives/features_macrostructure/<subtree>/{file_id}_sleep_metrics.tsv` | Long format: `group`, `metric`, `value`, `unit`, `definition_short` (metrics + provenance rows). The skip marker: delete it to reprocess that recording. | `df.pivot(index='file_id', columns='metric', values='value')` gives the wide row. |
| `derivatives/features_macrostructure/<subtree>/{file_id}_sleep_metrics_checks.tsv` | The recording's checks (same columns as the global checks file). | Written before the metrics file, so an interrupted run leaves no metrics without its checks. |